<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/06_LLM_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 06.01 — INITIALIZATION AND RESEARCH SCOPE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06")
print("CANDIDATE STRATEGY EVALUATION")
print("=" * 100)

print("\nResearch objective:")
print(
    "Empirically evaluate candidate missing-value imputation "
    "strategies under controlled missingness scenarios and "
    "generate evidence for AIR-LLM adaptive strategy selection."
)

print("\nResearch scope:")
print("  • Multiple benchmark datasets")
print("  • Feature-level evaluation")
print("  • MCAR and MAR missingness")
print("  • Missingness rates configured in Notebook 00")
print("  • Repeated deterministic experiments")
print("  • Ground-truth recovery from observed values")
print("  • Numerical reconstruction metrics")
print("  • Categorical reconstruction metrics")
print("  • Distributional fidelity")
print("  • Dependency preservation")
print("  • Downstream predictive utility")
print("  • Runtime measurement")
print("  • LLM recommendation evaluation")
print("  • Statistical significance analysis")
print("  • Adaptive utility-based selection")

print("\nNotebook sequence:")
for cell, purpose in [
    ("06.01", "Notebook 06 initialization and research scope"),
    ("06.02", "Load Notebook 00 master configuration"),
    ("06.03", "Validate datasets, features, targets, and experiment configuration"),
    ("06.04", "Load processed experimental datasets"),
    ("06.05", "Build evaluation dataset registry"),
    ("06.06", "Detect/validate feature types"),
    ("06.07", "Build predictor-selection function"),
    ("06.08", "Build missingness-mask generation"),
    ("06.09", "Build predictor encoding pipeline"),
    ("06.10", "Define statistical candidate imputers"),
    ("06.11", "Define KNN candidate"),
    ("06.12", "Define iterative/MICE candidates"),
    ("06.13", "Define Random Forest candidate"),
    ("06.14", "Define Gradient Boosting candidate"),
    ("06.15", "Define MissForest candidate"),
    ("06.16", "Define SoftImpute / matrix-factorization candidates"),
    ("06.17", "Candidate strategy compatibility validation"),
    ("06.18", "Build LLM-recommended candidate evaluation plan"),
    ("06.19", "Validate evaluation-plan integrity and expected experiment count"),
    ("06.20", "Small-run preparation / candidate execution preflight"),
    ("06.21", "Corrected Candidate Evaluation Engine + small validation + full execution"),
    ("06.22", "Evaluation Status Summary"),
    ("06.23", "Candidate Performance Analysis"),
    ("06.24", "LLM Recommendation Evaluation"),
    ("06.25", "Statistical Significance"),
    ("06.26", "Adaptive Utility-Based Selection"),
]:
    print(f"  {cell}  {purpose}")

print("\nResearch principle:")
print(
    "Observed values are masked to create known ground truth; "
    "imputation is performed without access to the masked truth."
)

print("\n" + "=" * 100)
print("NOTEBOOK 06 INITIALIZATION : READY")
print("=" * 100)

In [73]:
# ============================================================
# NOTEBOOK 06.02 — LOAD NOTEBOOK 00 MASTER CONFIGURATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.02")
print("LOAD NOTEBOOK 00 MASTER CONFIGURATION")
print("=" * 100)

import os
import json
import yaml
import random
import warnings
import time
import inspect
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Canonical AIR-LLM project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

CONFIG_PATH = (
    PROJECT_ROOT /
    "config" /
    "config.yaml"
)


if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"AIR-LLM project root not found:\n{PROJECT_ROOT}"
    )


if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 00 configuration not found:\n{CONFIG_PATH}"
    )


with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    CONFIG = yaml.safe_load(f)


if not isinstance(CONFIG, dict):
    raise TypeError(
        "config.yaml must contain a dictionary."
    )


print(f"\nProject root : {PROJECT_ROOT}")
print(f"Config file  : {CONFIG_PATH}")

print(
    f"Top-level configuration sections: "
    f"{len(CONFIG)}"
)

print("\nMaster configuration loaded successfully.")
print("=" * 100)

AIR-LLM — NOTEBOOK 06.02
LOAD NOTEBOOK 00 MASTER CONFIGURATION

Project root : /content/drive/MyDrive/AIR_LLM_Research
Config file  : /content/drive/MyDrive/AIR_LLM_Research/config/config.yaml
Top-level configuration sections: 25

Master configuration loaded successfully.


In [76]:
# ============================================================
# NOTEBOOK 06.03 — DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.03")
print("DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION")
print("=" * 100)


# ------------------------------------------------------------
# 1. Required state
# ------------------------------------------------------------

if "CONFIG" not in globals():
    raise RuntimeError(
        "CONFIG is not available. Run Notebook 06.02 first."
    )

if "EVALUATION_DATA" not in globals():
    raise RuntimeError(
        "EVALUATION_DATA is not available. Run Notebook 06.04 first."
    )

if not isinstance(EVALUATION_DATA, dict):
    raise RuntimeError(
        "EVALUATION_DATA must be a dictionary."
    )

if not EVALUATION_DATA:
    raise RuntimeError(
        "EVALUATION_DATA is empty."
    )


# ------------------------------------------------------------
# 2. Required AIR-LLM datasets
# ------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


available_datasets = {
    str(dataset_id)
    for dataset_id in EVALUATION_DATA.keys()
}


missing_datasets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in available_datasets
]


if missing_datasets:
    raise RuntimeError(
        "Required datasets are missing from EVALUATION_DATA:\n"
        +
        "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_datasets
        )
        +
        "\n\nAvailable datasets:\n"
        +
        "\n".join(
            f"  - {dataset_id}"
            for dataset_id in sorted(
                available_datasets
            )
        )
    )


# ------------------------------------------------------------
# 3. Resolve TARGET_REGISTRY
# ------------------------------------------------------------

target_registry = None


# Existing runtime registry has highest priority
if (
    "TARGET_REGISTRY" in globals()
    and isinstance(
        TARGET_REGISTRY,
        dict
    )
):

    target_registry = {
        str(k): str(v)
        for k, v in TARGET_REGISTRY.items()
        if v is not None
    }


# Try common configuration locations only if needed
if not target_registry:

    config_search = [
        CONFIG.get("TARGET_REGISTRY"),
        CONFIG.get("target_registry"),
        CONFIG.get("targets"),
    ]

    features_config = CONFIG.get(
        "features",
        {}
    )

    if isinstance(
        features_config,
        dict
    ):

        config_search.extend([
            features_config.get(
                "TARGET_REGISTRY"
            ),
            features_config.get(
                "target_registry"
            ),
            features_config.get(
                "targets"
            ),
        ])


    for candidate in config_search:

        if not isinstance(
            candidate,
            dict
        ):
            continue

        extracted = {}

        for dataset_id, value in candidate.items():

            if isinstance(
                value,
                str
            ):

                extracted[
                    str(dataset_id)
                ] = value

            elif isinstance(
                value,
                dict
            ):

                for key in [
                    "target",
                    "target_column",
                    "target_feature",
                    "column",
                ]:

                    if value.get(key) is not None:

                        extracted[
                            str(dataset_id)
                        ] = str(
                            value[key]
                        )

                        break

        if extracted:
            target_registry = extracted
            break


# ------------------------------------------------------------
# 4. If target registry is unavailable, recover targets
#    from the actual evaluation datasets/configuration
# ------------------------------------------------------------

if not target_registry:

    target_registry = {}


# Known AIR-LLM target definitions
KNOWN_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}


for dataset_id in EXPECTED_DATASETS:

    if dataset_id in target_registry:
        continue

    df = EVALUATION_DATA[
        dataset_id
    ]

    candidate = KNOWN_TARGETS.get(
        dataset_id
    )

    if (
        candidate is not None
        and candidate in df.columns
    ):

        target_registry[
            dataset_id
        ] = candidate


# ------------------------------------------------------------
# 5. Validate target definitions
# ------------------------------------------------------------

missing_targets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in target_registry
]


if missing_targets:

    raise RuntimeError(
        "Missing target definitions for:\n"
        +
        "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_targets
        )
    )


invalid_targets = []

for dataset_id in EXPECTED_DATASETS:

    target = target_registry[
        dataset_id
    ]

    if target not in EVALUATION_DATA[
        dataset_id
    ].columns:

        invalid_targets.append(
            f"{dataset_id} -> {target}"
        )


if invalid_targets:

    raise RuntimeError(
        "Target columns are not present in "
        "the corresponding evaluation datasets:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in invalid_targets
        )
    )


# Export validated registry
TARGET_REGISTRY = target_registry


# ------------------------------------------------------------
# 6. Validate experiment configuration
# ------------------------------------------------------------

missingness_config = CONFIG.get(
    "missingness",
    {}
)

if not isinstance(
    missingness_config,
    dict
):

    missingness_config = {}


# Missingness mechanisms
missingness_mechanisms = (
    missingness_config.get(
        "mechanisms"
    )
    or
    missingness_config.get(
        "missingness_mechanisms"
    )
    or
    CONFIG.get(
        "missingness_mechanisms"
    )
)


if missingness_mechanisms is None:

    missingness_mechanisms = [
        "MCAR",
        "MAR",
        "MNAR",
    ]


missingness_mechanisms = [
    str(x).upper()
    for x in missingness_mechanisms
]


invalid_mechanisms = (
    set(missingness_mechanisms)
    -
    {
        "MCAR",
        "MAR",
        "MNAR",
    }
)


if invalid_mechanisms:

    raise RuntimeError(
        "Invalid missingness mechanisms: "
        +
        ", ".join(
            sorted(
                invalid_mechanisms
            )
        )
    )


# Missingness rates
missingness_rates = (
    missingness_config.get(
        "rates"
    )
    or
    missingness_config.get(
        "missingness_rates"
    )
    or
    CONFIG.get(
        "missingness_rates"
    )
)


if missingness_rates is None:

    missingness_rates = [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ]


missingness_rates = [
    float(x)
    for x in missingness_rates
]


if not all(
    0 < x < 1
    for x in missingness_rates
):

    raise RuntimeError(
        "Invalid missingness rates: "
        f"{missingness_rates}"
    )


# Repetitions
repetitions = (
    missingness_config.get(
        "repetitions"
    )
    or
    CONFIG.get(
        "repetitions"
    )
)


if repetitions is None:

    repetitions = 5


repetitions = int(
    repetitions
)


if repetitions < 1:

    raise RuntimeError(
        "Repetitions must be >= 1."
    )


# Master seed
if "MASTER_SEED" in globals():

    master_seed = int(
        MASTER_SEED
    )

else:

    master_seed = (
        CONFIG.get(
            "master_seed"
        )
        or
        CONFIG.get(
            "seed"
        )
        or
        42
    )

    master_seed = int(
        master_seed
    )


MASTER_SEED = master_seed


# ------------------------------------------------------------
# 7. Dataset structure validation
# ------------------------------------------------------------

DATASET_VALIDATION_ROWS = []

for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    if not isinstance(
        df,
        pd.DataFrame
    ):

        raise RuntimeError(
            f"{dataset_id} is not a pandas DataFrame."
        )

    target = TARGET_REGISTRY[
        dataset_id
    ]

    DATASET_VALIDATION_ROWS.append({

        "dataset_id":
            dataset_id,

        "rows":
            len(df),

        "columns":
            df.shape[1],

        "target":
            target,

        "target_present":
            target in df.columns,

        "missing_cells":
            int(
                df.isna().sum().sum()
            ),
    })


DATASET_FEATURE_TARGET_VALIDATION_DF = pd.DataFrame(
    DATASET_VALIDATION_ROWS
)


display(
    DATASET_FEATURE_TARGET_VALIDATION_DF
)


# ------------------------------------------------------------
# 8. Export experiment configuration
# ------------------------------------------------------------

EXPERIMENT_CONFIGURATION = {

    "datasets":
        EXPECTED_DATASETS,

    "targets":
        TARGET_REGISTRY,

    "missingness_mechanisms":
        missingness_mechanisms,

    "missingness_rates":
        missingness_rates,

    "repetitions":
        repetitions,

    "master_seed":
        MASTER_SEED,
}


# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

print("\nDATASETS")
print("-" * 100)

for dataset_id in EXPECTED_DATASETS:

    print(
        f"  {dataset_id}"
    )


print("\nTARGETS")
print("-" * 100)

for dataset_id in EXPECTED_DATASETS:

    print(
        f"  {dataset_id} -> "
        f"{TARGET_REGISTRY[dataset_id]}"
    )


print("\nEXPERIMENT CONFIGURATION")
print("-" * 100)

print(
    "  Missingness mechanisms : "
    f"{missingness_mechanisms}"
)

print(
    "  Missingness rates      : "
    f"{missingness_rates}"
)

print(
    "  Repetitions            : "
    f"{repetitions}"
)

print(
    "  Master seed            : "
    f"{MASTER_SEED}"
)


print("\n" + "=" * 100)
print(
    "DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.03
DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION


,dataset_id,rows,columns,target,target_present,missing_cells
0,adult_income,5000,15,income,True,727
1,bank_marketing,5000,17,y,True,0
2,diabetes_130us,5000,48,readmitted,True,18329



DATASETS
----------------------------------------------------------------------------------------------------
  adult_income
  bank_marketing
  diabetes_130us

TARGETS
----------------------------------------------------------------------------------------------------
  adult_income -> income
  bank_marketing -> y
  diabetes_130us -> readmitted

EXPERIMENT CONFIGURATION
----------------------------------------------------------------------------------------------------
  Missingness mechanisms : ['MCAR', 'MAR', 'MNAR']
  Missingness rates      : [0.1, 0.2, 0.3, 0.4, 0.5]
  Repetitions            : 5
  Master seed            : 42

DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION : PASSED


In [77]:
# ============================================================
# NOTEBOOK 06.03 — DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.03")
print("DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION")
print("=" * 100)


# ------------------------------------------------------------
# 1. Required state
# ------------------------------------------------------------

if "CONFIG" not in globals():
    raise RuntimeError(
        "CONFIG is not available. Run Notebook 06.02 first."
    )

if "EVALUATION_DATA" not in globals():
    raise RuntimeError(
        "EVALUATION_DATA is not available. Run Notebook 06.04 first."
    )

if not isinstance(EVALUATION_DATA, dict):
    raise RuntimeError(
        "EVALUATION_DATA must be a dictionary."
    )

if not EVALUATION_DATA:
    raise RuntimeError(
        "EVALUATION_DATA is empty."
    )


# ------------------------------------------------------------
# 2. Required AIR-LLM datasets
# ------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


available_datasets = {
    str(dataset_id)
    for dataset_id in EVALUATION_DATA.keys()
}


missing_datasets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in available_datasets
]


if missing_datasets:
    raise RuntimeError(
        "Required datasets are missing from EVALUATION_DATA:\n"
        +
        "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_datasets
        )
        +
        "\n\nAvailable datasets:\n"
        +
        "\n".join(
            f"  - {dataset_id}"
            for dataset_id in sorted(
                available_datasets
            )
        )
    )


# ------------------------------------------------------------
# 3. Resolve TARGET_REGISTRY
# ------------------------------------------------------------

target_registry = None


# Existing runtime registry has highest priority
if (
    "TARGET_REGISTRY" in globals()
    and isinstance(
        TARGET_REGISTRY,
        dict
    )
):

    target_registry = {
        str(k): str(v)
        for k, v in TARGET_REGISTRY.items()
        if v is not None
    }


# Try common configuration locations only if needed
if not target_registry:

    config_search = [
        CONFIG.get("TARGET_REGISTRY"),
        CONFIG.get("target_registry"),
        CONFIG.get("targets"),
    ]

    features_config = CONFIG.get(
        "features",
        {}
    )

    if isinstance(
        features_config,
        dict
    ):

        config_search.extend([
            features_config.get(
                "TARGET_REGISTRY"
            ),
            features_config.get(
                "target_registry"
            ),
            features_config.get(
                "targets"
            ),
        ])


    for candidate in config_search:

        if not isinstance(
            candidate,
            dict
        ):
            continue

        extracted = {}

        for dataset_id, value in candidate.items():

            if isinstance(
                value,
                str
            ):

                extracted[
                    str(dataset_id)
                ] = value

            elif isinstance(
                value,
                dict
            ):

                for key in [
                    "target",
                    "target_column",
                    "target_feature",
                    "column",
                ]:

                    if value.get(key) is not None:

                        extracted[
                            str(dataset_id)
                        ] = str(
                            value[key]
                        )

                        break

        if extracted:
            target_registry = extracted
            break


# ------------------------------------------------------------
# 4. If target registry is unavailable, recover targets
#    from the actual evaluation datasets/configuration
# ------------------------------------------------------------

if not target_registry:

    target_registry = {}


# Known AIR-LLM target definitions
KNOWN_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}


for dataset_id in EXPECTED_DATASETS:

    if dataset_id in target_registry:
        continue

    df = EVALUATION_DATA[
        dataset_id
    ]

    candidate = KNOWN_TARGETS.get(
        dataset_id
    )

    if (
        candidate is not None
        and candidate in df.columns
    ):

        target_registry[
            dataset_id
        ] = candidate


# ------------------------------------------------------------
# 5. Validate target definitions
# ------------------------------------------------------------

missing_targets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in target_registry
]


if missing_targets:

    raise RuntimeError(
        "Missing target definitions for:\n"
        +
        "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_targets
        )
    )


invalid_targets = []

for dataset_id in EXPECTED_DATASETS:

    target = target_registry[
        dataset_id
    ]

    if target not in EVALUATION_DATA[
        dataset_id
    ].columns:

        invalid_targets.append(
            f"{dataset_id} -> {target}"
        )


if invalid_targets:

    raise RuntimeError(
        "Target columns are not present in "
        "the corresponding evaluation datasets:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in invalid_targets
        )
    )


# Export validated registry
TARGET_REGISTRY = target_registry


# ------------------------------------------------------------
# 6. Validate experiment configuration
# ------------------------------------------------------------

missingness_config = CONFIG.get(
    "missingness",
    {}
)

if not isinstance(
    missingness_config,
    dict
):

    missingness_config = {}


# Missingness mechanisms
missingness_mechanisms = (
    missingness_config.get(
        "mechanisms"
    )
    or
    missingness_config.get(
        "missingness_mechanisms"
    )
    or
    CONFIG.get(
        "missingness_mechanisms"
    )
)


if missingness_mechanisms is None:

    missingness_mechanisms = [
        "MCAR",
        "MAR",
        "MNAR",
    ]


missingness_mechanisms = [
    str(x).upper()
    for x in missingness_mechanisms
]


invalid_mechanisms = (
    set(missingness_mechanisms)
    -
    {
        "MCAR",
        "MAR",
        "MNAR",
    }
)


if invalid_mechanisms:

    raise RuntimeError(
        "Invalid missingness mechanisms: "
        +
        ", ".join(
            sorted(
                invalid_mechanisms
            )
        )
    )


# Missingness rates
missingness_rates = (
    missingness_config.get(
        "rates"
    )
    or
    missingness_config.get(
        "missingness_rates"
    )
    or
    CONFIG.get(
        "missingness_rates"
    )
)


if missingness_rates is None:

    missingness_rates = [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ]


missingness_rates = [
    float(x)
    for x in missingness_rates
]


if not all(
    0 < x < 1
    for x in missingness_rates
):

    raise RuntimeError(
        "Invalid missingness rates: "
        f"{missingness_rates}"
    )


# Repetitions
repetitions = (
    missingness_config.get(
        "repetitions"
    )
    or
    CONFIG.get(
        "repetitions"
    )
)


if repetitions is None:

    repetitions = 5


repetitions = int(
    repetitions
)


if repetitions < 1:

    raise RuntimeError(
        "Repetitions must be >= 1."
    )


# Master seed
if "MASTER_SEED" in globals():

    master_seed = int(
        MASTER_SEED
    )

else:

    master_seed = (
        CONFIG.get(
            "master_seed"
        )
        or
        CONFIG.get(
            "seed"
        )
        or
        42
    )

    master_seed = int(
        master_seed
    )


MASTER_SEED = master_seed


# ------------------------------------------------------------
# 7. Dataset structure validation
# ------------------------------------------------------------

DATASET_VALIDATION_ROWS = []

for dataset_id in EXPECTED_DATASETS:

    df = EVALUATION_DATA[
        dataset_id
    ]

    if not isinstance(
        df,
        pd.DataFrame
    ):

        raise RuntimeError(
            f"{dataset_id} is not a pandas DataFrame."
        )

    target = TARGET_REGISTRY[
        dataset_id
    ]

    DATASET_VALIDATION_ROWS.append({

        "dataset_id":
            dataset_id,

        "rows":
            len(df),

        "columns":
            df.shape[1],

        "target":
            target,

        "target_present":
            target in df.columns,

        "missing_cells":
            int(
                df.isna().sum().sum()
            ),
    })


DATASET_FEATURE_TARGET_VALIDATION_DF = pd.DataFrame(
    DATASET_VALIDATION_ROWS
)


display(
    DATASET_FEATURE_TARGET_VALIDATION_DF
)


# ------------------------------------------------------------
# 8. Export experiment configuration
# ------------------------------------------------------------

EXPERIMENT_CONFIGURATION = {

    "datasets":
        EXPECTED_DATASETS,

    "targets":
        TARGET_REGISTRY,

    "missingness_mechanisms":
        missingness_mechanisms,

    "missingness_rates":
        missingness_rates,

    "repetitions":
        repetitions,

    "master_seed":
        MASTER_SEED,
}


# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

print("\nDATASETS")
print("-" * 100)

for dataset_id in EXPECTED_DATASETS:

    print(
        f"  {dataset_id}"
    )


print("\nTARGETS")
print("-" * 100)

for dataset_id in EXPECTED_DATASETS:

    print(
        f"  {dataset_id} -> "
        f"{TARGET_REGISTRY[dataset_id]}"
    )


print("\nEXPERIMENT CONFIGURATION")
print("-" * 100)

print(
    "  Missingness mechanisms : "
    f"{missingness_mechanisms}"
)

print(
    "  Missingness rates      : "
    f"{missingness_rates}"
)

print(
    "  Repetitions            : "
    f"{repetitions}"
)

print(
    "  Master seed            : "
    f"{MASTER_SEED}"
)


print("\n" + "=" * 100)
print(
    "DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION : PASSED"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.03
DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION


,dataset_id,rows,columns,target,target_present,missing_cells
0,adult_income,5000,15,income,True,727
1,bank_marketing,5000,17,y,True,0
2,diabetes_130us,5000,48,readmitted,True,18329



DATASETS
----------------------------------------------------------------------------------------------------
  adult_income
  bank_marketing
  diabetes_130us

TARGETS
----------------------------------------------------------------------------------------------------
  adult_income -> income
  bank_marketing -> y
  diabetes_130us -> readmitted

EXPERIMENT CONFIGURATION
----------------------------------------------------------------------------------------------------
  Missingness mechanisms : ['MCAR', 'MAR', 'MNAR']
  Missingness rates      : [0.1, 0.2, 0.3, 0.4, 0.5]
  Repetitions            : 5
  Master seed            : 42

DATASET / FEATURE / TARGET / EXPERIMENT VALIDATION : PASSED


In [78]:
# ============================================================
# NOTEBOOK 06.04 — LOAD PROCESSED EXPERIMENTAL DATASETS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.04")
print("LOAD PROCESSED EXPERIMENTAL DATASETS")
print("=" * 100)


PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)


if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed-data directory not found:\n{PROCESSED_DIR}"
    )


def find_processed_dataset(dataset_id):

    candidates = [
        PROCESSED_DIR / f"{dataset_id}_processed.csv",
        PROCESSED_DIR / f"{dataset_id}.csv",
        PROCESSED_DIR / dataset_id / "processed.csv",
    ]

    for path in candidates:
        if path.exists():
            return path

    recursive_matches = list(
        PROCESSED_DIR.rglob(
            f"{dataset_id}*.csv"
        )
    )

    if recursive_matches:
        return recursive_matches[0]

    return None


EVALUATION_DATA = {}
DATASET_PATH_REGISTRY = {}

for dataset_id in DATASET_IDS:

    path = find_processed_dataset(
        dataset_id
    )

    if path is None:
        raise FileNotFoundError(
            f"No processed dataset found for '{dataset_id}'."
        )

    df = pd.read_csv(path)

    if df.empty:
        raise RuntimeError(
            f"Processed dataset '{dataset_id}' is empty."
        )

    EVALUATION_DATA[dataset_id] = df
    DATASET_PATH_REGISTRY[dataset_id] = path

    print(
        f"{dataset_id:<20} "
        f"rows={len(df):>8,} "
        f"columns={df.shape[1]:>4} "
        f"path={path}"
    )


print(
    f"\nDatasets loaded: {len(EVALUATION_DATA)}"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.04
LOAD PROCESSED EXPERIMENTAL DATASETS
adult_income         rows=  32,561 columns=  15 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/adult_income_processed.csv
bank_marketing       rows=  45,211 columns=  17 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/bank_marketing_processed.csv
diabetes_130us       rows= 101,766 columns=  48 path=/content/drive/MyDrive/AIR_LLM_Research/data/processed/diabetes_130us_processed.csv

Datasets loaded: 3


In [79]:
# ============================================================
# NOTEBOOK 06.06 — FEATURE TYPE DETECTION / VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.06")
print("FEATURE TYPE DETECTION / VALIDATION")
print("=" * 100)


def detect_feature_type(series):

    if pd.api.types.is_bool_dtype(series):
        return "categorical"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    return "categorical"


FEATURE_TYPE_REGISTRY = {}
feature_type_rows = []


for dataset_id, df in EVALUATION_DATA.items():

    target = TARGET_REGISTRY[dataset_id]

    FEATURE_TYPE_REGISTRY[dataset_id] = {}

    for feature in df.columns:

        feature_type = detect_feature_type(
            df[feature]
        )

        FEATURE_TYPE_REGISTRY[
            dataset_id
        ][feature] = feature_type

        feature_type_rows.append({
            "dataset_id": dataset_id,
            "feature": feature,
            "feature_type": feature_type,
            "is_target": feature == target,
            "unique_values": int(
                df[feature].nunique(
                    dropna=True
                )
            )
        })


FEATURE_TYPE_DF = pd.DataFrame(
    feature_type_rows
)


if FEATURE_TYPE_DF.empty:
    raise RuntimeError(
        "Feature-type registry is empty."
    )


display(
    FEATURE_TYPE_DF
)


print(
    "\nFeature-type detection: PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.06
FEATURE TYPE DETECTION / VALIDATION


,dataset_id,feature,feature_type,is_target,unique_values
0,adult_income,age,numerical,False,73
1,adult_income,workclass,categorical,False,8
2,adult_income,fnlwgt,numerical,False,21648
3,adult_income,education,categorical,False,16
4,adult_income,education_num,numerical,False,16
...,...,...,...,...,...
75,diabetes_130us,metformin-rosiglitazone,categorical,False,2
76,diabetes_130us,metformin-pioglitazone,categorical,False,2
77,diabetes_130us,change,categorical,False,2
78,diabetes_130us,diabetesMed,categorical,False,2



Feature-type detection: PASSED


In [80]:
# ============================================================
# NOTEBOOK 06.07 — PREDICTOR SELECTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.07")
print("BUILD PREDICTOR-SELECTION FUNCTION")
print("=" * 100)


def select_predictors(
    df,
    target_feature,
    candidate_feature=None,
    max_predictors=20
):

    if target_feature not in df.columns:
        raise KeyError(
            f"Target '{target_feature}' not found."
        )

    excluded = {
        target_feature
    }

    if candidate_feature is not None:
        excluded.add(candidate_feature)

    predictors = [
        column
        for column in df.columns
        if column not in excluded
    ]

    if not predictors:
        return []

    # Prefer predictors with lower missingness.
    missing_rates = (
        df[predictors]
        .isna()
        .mean()
        .sort_values()
    )

    predictors = (
        missing_rates
        .head(max_predictors)
        .index
        .tolist()
    )

    return predictors


print(
    "Predictor-selection function: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.07
BUILD PREDICTOR-SELECTION FUNCTION
Predictor-selection function: READY


In [81]:
# ============================================================
# NOTEBOOK 06.08 — MISSINGNESS MASK GENERATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.08")
print("BUILD MISSINGNESS-MASK GENERATION")
print("=" * 100)


def generate_missingness_mask(
    series,
    rate,
    mechanism="MCAR",
    seed=42,
    predictors=None
):

    mechanism = str(
        mechanism
    ).upper()

    if mechanism not in {
        "MCAR",
        "MAR"
    }:
        raise ValueError(
            "Supported mechanisms are MCAR and MAR."
        )

    if not 0 < rate < 1:
        raise ValueError(
            "Missingness rate must satisfy 0 < rate < 1."
        )

    rng = np.random.default_rng(
        int(seed)
    )

    observed = series.notna().to_numpy()

    candidate_indices = np.where(
        observed
    )[0]

    if len(candidate_indices) == 0:
        return pd.Series(
            False,
            index=series.index
        )

    n_remove = int(
        np.floor(
            len(candidate_indices) * rate
        )
    )

    n_remove = max(
        1,
        min(
            n_remove,
            len(candidate_indices)
        )
    )

    if mechanism == "MCAR" or predictors is None:
        selected = rng.choice(
            candidate_indices,
            size=n_remove,
            replace=False
        )

    else:
        # MAR proxy:
        # use predictor-derived propensity scores.
        predictor_df = predictors.copy()

        numeric = predictor_df.select_dtypes(
            include=[np.number]
        )

        if numeric.shape[1] == 0:
            selected = rng.choice(
                candidate_indices,
                size=n_remove,
                replace=False
            )

        else:
            score = (
                numeric
                .rank(
                    pct=True
                )
                .mean(
                    axis=1
                )
                .fillna(0.5)
                .to_numpy()
            )

            score = score[candidate_indices]

            order = np.argsort(
                score
                + rng.uniform(
                    0,
                    1e-9,
                    size=len(score)
                )
            )

            selected = candidate_indices[
                order[-n_remove:]
            ]

    mask = pd.Series(
        False,
        index=series.index
    )

    mask.iloc[selected] = True

    return mask


print(
    "Missingness-mask generator: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.08
BUILD MISSINGNESS-MASK GENERATION
Missingness-mask generator: READY


In [82]:
# ============================================================
# NOTEBOOK 06.09 — PREDICTOR ENCODING PIPELINE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.09")
print("BUILD PREDICTOR ENCODING PIPELINE")
print("=" * 100)


from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)


def encode_predictors(
    X_train,
    X_test
):

    X_train = X_train.copy()
    X_test = X_test.copy()

    numerical_columns = (
        X_train
        .select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    categorical_columns = [
        column
        for column in X_train.columns
        if column not in numerical_columns
    ]

    transformers = []

    if numerical_columns:

        transformers.append(
            (
                "numerical",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    ),
                    (
                        "scaler",
                        StandardScaler()
                    )
                ]),
                numerical_columns
            )
        )

    if categorical_columns:

        transformers.append(
            (
                "categorical",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent"
                        )
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False
                        )
                    )
                ]),
                categorical_columns
            )
        )

    if not transformers:
        raise ValueError(
            "No predictor columns available."
        )

    predictor_encoder = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    X_train_encoded = (
        predictor_encoder
        .fit_transform(X_train)
    )

    X_test_encoded = (
        predictor_encoder
        .transform(X_test)
    )

    predictor_scaler = None

    return (
        np.asarray(
            X_train_encoded,
            dtype=float
        ),
        np.asarray(
            X_test_encoded,
            dtype=float
        ),
        predictor_encoder,
        predictor_scaler
    )


print(
    "Predictor encoder: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.09
BUILD PREDICTOR ENCODING PIPELINE
Predictor encoder: READY


In [83]:
# ============================================================
# NOTEBOOK 06.10 — STATISTICAL CANDIDATE IMPUTERS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.10")
print("STATISTICAL CANDIDATE IMPUTERS")
print("=" * 100)


def impute_mean(
    train_series,
    test_series=None
):

    value = train_series.mean()

    if pd.isna(value):
        value = 0.0

    result = train_series.copy()
    result = result.fillna(value)

    return result


def impute_median(
    train_series,
    test_series=None
):

    value = train_series.median()

    if pd.isna(value):
        value = 0.0

    result = train_series.copy()
    result = result.fillna(value)

    return result


def impute_mode(
    train_series,
    test_series=None
):

    mode = train_series.mode(
        dropna=True
    )

    value = (
        mode.iloc[0]
        if not mode.empty
        else 0
    )

    result = train_series.copy()
    result = result.fillna(value)

    return result


def impute_constant(
    train_series,
    test_series=None,
    constant_value=None
):

    result = train_series.copy()

    if constant_value is None:

        if pd.api.types.is_numeric_dtype(
            train_series
        ):
            constant_value = 0.0

        else:
            constant_value = "MISSING"

    result = result.fillna(
        constant_value
    )

    return result


def impute_random_sample(
    train_series,
    test_series=None,
    random_state=42
):

    result = train_series.copy()

    observed = (
        train_series
        .dropna()
        .to_numpy()
    )

    if len(observed) == 0:
        return result.fillna(0)

    missing_count = int(
        result.isna().sum()
    )

    rng = np.random.default_rng(
        random_state
    )

    sampled = rng.choice(
        observed,
        size=missing_count,
        replace=True
    )

    result.loc[
        result.isna()
    ] = sampled

    return result


print(
    "Statistical candidates: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.10
STATISTICAL CANDIDATE IMPUTERS
Statistical candidates: READY


In [84]:
# ============================================================
# NOTEBOOK 06.11 — KNN IMPUTATION CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.11")
print("KNN IMPUTATION CANDIDATE")
print("=" * 100)


from sklearn.impute import KNNImputer


def fit_knn(
    X_train,
    X_test=None,
    n_neighbors=5
):

    imputer = KNNImputer(
        n_neighbors=n_neighbors
    )

    X_train_imputed = (
        imputer.fit_transform(
            X_train
        )
    )

    if X_test is not None:

        X_test_imputed = (
            imputer.transform(
                X_test
            )
        )

    else:

        X_test_imputed = None

    return (
        X_train_imputed,
        X_test_imputed,
        imputer
    )


print(
    "KNN candidate: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.11
KNN IMPUTATION CANDIDATE
KNN candidate: READY


In [85]:
# ============================================================
# NOTEBOOK 06.12 — ITERATIVE / MICE CANDIDATES
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.12")
print("ITERATIVE / MICE IMPUTATION CANDIDATES")
print("=" * 100)


from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


def fit_iterative(
    X_train,
    X_test=None,
    estimator=None,
    random_state=42,
    max_iter=10
):

    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=max_iter,
        random_state=random_state,
        initial_strategy="median",
        skip_complete=True
    )

    X_train_imputed = (
        imputer.fit_transform(
            X_train
        )
    )

    X_test_imputed = (
        imputer.transform(X_test)
        if X_test is not None
        else None
    )

    return (
        X_train_imputed,
        X_test_imputed,
        imputer
    )


def fit_mice(
    X_train,
    X_test=None,
    random_state=42,
    max_iter=10
):

    return fit_iterative(
        X_train=X_train,
        X_test=X_test,
        estimator=None,
        random_state=random_state,
        max_iter=max_iter
    )


print(
    "Iterative candidate: READY"
)

print(
    "MICE candidate: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.12
ITERATIVE / MICE IMPUTATION CANDIDATES
Iterative candidate: READY
MICE candidate: READY


In [86]:
# ============================================================
# NOTEBOOK 06.13 — RANDOM FOREST CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.13")
print("RANDOM FOREST IMPUTATION CANDIDATE")
print("=" * 100)


from sklearn.ensemble import RandomForestRegressor


def fit_random_forest(
    X_train,
    target_train,
    X_missing,
    feature_type="numerical",
    random_state=42
):

    if feature_type != "numerical":
        raise ValueError(
            "Random Forest candidate currently expects "
            "a numerical target representation."
        )

    observed_mask = target_train.notna()

    if observed_mask.sum() < 5:
        raise ValueError(
            "Insufficient observed target values "
            "for Random Forest imputation."
        )

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=random_state,
        n_jobs=-1,
        max_depth=None
    )

    model.fit(
        X_train.loc[observed_mask],
        target_train.loc[observed_mask]
    )

    result = target_train.copy()

    missing_mask = target_train.isna()

    if missing_mask.any():

        result.loc[missing_mask] = (
            model.predict(
                X_missing.loc[missing_mask]
            )
        )

    return (
        result,
        model
    )


print(
    "Random Forest candidate: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.13
RANDOM FOREST IMPUTATION CANDIDATE
Random Forest candidate: READY


In [87]:
# ============================================================
# NOTEBOOK 06.14 — GRADIENT BOOSTING CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.14")
print("GRADIENT BOOSTING IMPUTATION CANDIDATE")
print("=" * 100)


from sklearn.ensemble import HistGradientBoostingRegressor


def fit_gradient_boosting(
    X_train,
    target_train,
    X_missing,
    feature_type="numerical",
    random_state=42
):

    if feature_type != "numerical":
        raise ValueError(
            "Gradient Boosting candidate currently expects "
            "a numerical target representation."
        )

    observed_mask = target_train.notna()

    if observed_mask.sum() < 5:
        raise ValueError(
            "Insufficient observed target values "
            "for Gradient Boosting imputation."
        )

    model = HistGradientBoostingRegressor(
        max_iter=100,
        learning_rate=0.05,
        random_state=random_state
    )

    model.fit(
        X_train.loc[observed_mask],
        target_train.loc[observed_mask]
    )

    result = target_train.copy()

    missing_mask = target_train.isna()

    if missing_mask.any():

        result.loc[missing_mask] = (
            model.predict(
                X_missing.loc[missing_mask]
            )
        )

    return (
        result,
        model
    )


print(
    "Gradient Boosting candidate: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.14
GRADIENT BOOSTING IMPUTATION CANDIDATE
Gradient Boosting candidate: READY


In [88]:
# ============================================================
# NOTEBOOK 06.15 — MISSFOREST CANDIDATE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.15")
print("MISSFOREST IMPUTATION CANDIDATE")
print("=" * 100)


def fit_missforest(
    X_train,
    X_test=None,
    random_state=42,
    max_iter=5
):

    # Iterative Random-Forest approximation of MissForest.
    # Uses RandomForest estimators within IterativeImputer.

    estimator = RandomForestRegressor(
        n_estimators=50,
        random_state=random_state,
        n_jobs=-1,
        max_depth=None
    )

    imputer = IterativeImputer(
        estimator=estimator,
        max_iter=max_iter,
        random_state=random_state,
        initial_strategy="median",
        skip_complete=True
    )

    X_train_imputed = (
        imputer.fit_transform(
            X_train
        )
    )

    X_test_imputed = (
        imputer.transform(
            X_test
        )
        if X_test is not None
        else None
    )

    return (
        X_train_imputed,
        X_test_imputed,
        imputer
    )


print(
    "MissForest candidate: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.15
MISSFOREST IMPUTATION CANDIDATE
MissForest candidate: READY


In [89]:
# ============================================================
# NOTEBOOK 06.16 — SOFTIMPUTE / MATRIX FACTORIZATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.16")
print("SOFTIMPUTE / MATRIX-FACTORIZATION CANDIDATES")
print("=" * 100)


def fit_softimpute(
    X_train,
    X_test=None,
    rank=10,
    max_iter=50,
    tolerance=1e-4
):

    X = np.asarray(
        X_train,
        dtype=float
    )

    column_means = np.nanmean(
        X,
        axis=0
    )

    column_means = np.where(
        np.isfinite(column_means),
        column_means,
        0.0
    )

    filled = np.where(
        np.isnan(X),
        column_means,
        X
    )

    for _ in range(max_iter):

        previous = filled.copy()

        U, S, VT = np.linalg.svd(
            filled,
            full_matrices=False
        )

        k = min(
            rank,
            len(S)
        )

        reconstructed = (
            U[:, :k]
            @ np.diag(S[:k])
            @ VT[:k, :]
        )

        filled[np.isnan(X)] = (
            reconstructed[
                np.isnan(X)
            ]
        )

        observed_difference = (
            np.linalg.norm(
                filled - previous
            )
        )

        if observed_difference < tolerance:
            break

    X_test_imputed = None

    if X_test is not None:

        X_test_array = np.asarray(
            X_test,
            dtype=float
        )

        X_test_imputed = np.where(
            np.isnan(X_test_array),
            column_means,
            X_test_array
        )

    return (
        filled,
        X_test_imputed,
        {
            "rank": k,
            "iterations": _ + 1
        }
    )


def fit_matrix_factorization(
    X_train,
    X_test=None,
    rank=10,
    max_iter=50
):

    return fit_softimpute(
        X_train=X_train,
        X_test=X_test,
        rank=rank,
        max_iter=max_iter
    )


print(
    "SoftImpute candidate: READY"
)

print(
    "Matrix-factorization candidate: READY"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.16
SOFTIMPUTE / MATRIX-FACTORIZATION CANDIDATES
SoftImpute candidate: READY
Matrix-factorization candidate: READY


In [91]:
# ============================================================
# NOTEBOOK 06.17 — CANDIDATE STRATEGY COMPATIBILITY VALIDATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.17")
print("CANDIDATE STRATEGY COMPATIBILITY VALIDATION")
print("=" * 100)


# ------------------------------------------------------------
# 1. Required state
# ------------------------------------------------------------

if "STRATEGY_REGISTRY" not in globals():
    raise RuntimeError(
        "STRATEGY_REGISTRY is not available."
    )

if "FEATURE_TYPE_DF" not in globals():
    # Accept the actual dataframe created by 06.06
    if "FEATURE_TYPE_VALIDATION_DF" in globals():
        FEATURE_TYPE_DF = FEATURE_TYPE_VALIDATION_DF
    elif "FEATURE_PROFILE_DF" in globals():
        FEATURE_TYPE_DF = FEATURE_PROFILE_DF
    else:
        raise RuntimeError(
            "Feature-type validation dataframe is not available. "
            "Run Notebook 06.06 first."
        )


# ------------------------------------------------------------
# 2. Normalize feature-type values
# ------------------------------------------------------------

def _normalize_feature_type(value):

    value = str(value).strip().lower()

    if value in {
        "numeric",
        "numerical",
        "float",
        "float64",
        "float32",
        "integer",
        "int",
        "int64",
        "int32",
        "continuous"
    }:
        return "numerical"

    if value in {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "boolean",
        "bool"
    }:
        return "categorical"

    return value


# ------------------------------------------------------------
# 3. Resolve strategy feature-type metadata
# ------------------------------------------------------------

def _extract_strategy_feature_types(metadata):

    if not isinstance(metadata, dict):
        return set()

    possible_keys = [
        "feature_types",
        "supported_feature_types",
        "supported_types",
        "compatible_feature_types",
        "feature_type",
        "type",
    ]

    raw_value = None

    for key in possible_keys:

        if key in metadata:
            raw_value = metadata[key]

            if raw_value is not None:
                break

    if raw_value is None:
        return set()

    if isinstance(
        raw_value,
        str
    ):

        text = raw_value.strip()

        try:
            parsed = ast.literal_eval(text)

            if isinstance(
                parsed,
                (list, tuple, set)
            ):
                raw_values = list(parsed)
            else:
                raw_values = [parsed]

        except Exception:

            raw_values = [
                x.strip()
                for x in text.split(",")
                if x.strip()
            ]

    elif isinstance(
        raw_value,
        (list, tuple, set)
    ):

        raw_values = list(raw_value)

    else:

        raw_values = [raw_value]

    return {
        _normalize_feature_type(x)
        for x in raw_values
        if x is not None
    }


# ------------------------------------------------------------
# 4. Build normalized strategy compatibility table
# ------------------------------------------------------------

strategy_rows = []

for strategy_id, metadata in STRATEGY_REGISTRY.items():

    normalized_id = str(
        strategy_id
    ).strip().lower()

    supported_types = (
        _extract_strategy_feature_types(
            metadata
        )
    )

    # --------------------------------------------------------
    # Explicit AIR-LLM compatibility fallback
    # --------------------------------------------------------

    if not supported_types:

        explicit_compatibility = {

            "mean": {
                "numerical"
            },

            "median": {
                "numerical"
            },

            "mode": {
                "categorical"
            },

            "constant": {
                "numerical",
                "categorical"
            },

            "random_sample": {
                "numerical",
                "categorical"
            },

            "knn": {
                "numerical"
            },

            "iterative": {
                "numerical"
            },

            "mice": {
                "numerical",
                "categorical"
            },

            "random_forest": {
                "numerical",
                "categorical"
            },

            "gradient_boosting": {
                "numerical",
                "categorical"
            },

            "missforest": {
                "numerical",
                "categorical"
            },

            "softimpute": {
                "numerical"
            },

            "matrix_factorization": {
                "numerical"
            },
        }

        supported_types = (
            explicit_compatibility.get(
                normalized_id,
                set()
            )
        )

    if not supported_types:

        raise RuntimeError(
            f"Strategy '{strategy_id}' has no valid "
            "feature-type compatibility definition."
        )

    strategy_rows.append({

        "strategy_id":
            normalized_id,

        "supported_feature_types":
            sorted(
                supported_types
            ),

    })


STRATEGY_COMPATIBILITY_DF = pd.DataFrame(
    strategy_rows
)


# ------------------------------------------------------------
# 5. Validate all detected feature types
# ------------------------------------------------------------

if "feature_type" not in FEATURE_TYPE_DF.columns:

    raise RuntimeError(
        "Feature-type dataframe does not contain "
        "'feature_type'."
    )


detected_types = {
    _normalize_feature_type(x)
    for x in FEATURE_TYPE_DF[
        "feature_type"
    ].dropna().unique()
}


unsupported_detected_types = (
    detected_types
    -
    {
        "numerical",
        "categorical",
    }
)


if unsupported_detected_types:

    raise RuntimeError(
        "Unsupported detected feature types: "
        +
        ", ".join(
            sorted(
                unsupported_detected_types
            )
        )
    )


# ------------------------------------------------------------
# 6. Build compatibility validation
# ------------------------------------------------------------

compatibility_rows = []

for _, row in STRATEGY_COMPATIBILITY_DF.iterrows():

    supported = set(
        row[
            "supported_feature_types"
        ]
    )

    compatible_detected = (
        detected_types
        &
        supported
    )

    compatibility_rows.append({

        "strategy_id":
            row["strategy_id"],

        "supported_feature_types":
            sorted(
                supported
            ),

        "detected_compatible_types":
            sorted(
                compatible_detected
            ),

        "compatible":
            len(
                compatible_detected
            ) > 0,

    })


STRATEGY_COMPATIBILITY_VALIDATION_DF = pd.DataFrame(
    compatibility_rows
)


display(
    STRATEGY_COMPATIBILITY_VALIDATION_DF
)


# ------------------------------------------------------------
# 7. Validate strategy coverage
# ------------------------------------------------------------

invalid_strategies = (
    STRATEGY_COMPATIBILITY_VALIDATION_DF[
        ~STRATEGY_COMPATIBILITY_VALIDATION_DF[
            "compatible"
        ]
    ]["strategy_id"]
    .tolist()
)


if invalid_strategies:

    raise RuntimeError(
        "Strategies with no compatible feature types:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in invalid_strategies
        )
    )


# ------------------------------------------------------------
# 8. Create lookup used by evaluation engine
# ------------------------------------------------------------

STRATEGY_FEATURE_COMPATIBILITY = {

    row["strategy_id"]: set(
        row[
            "supported_feature_types"
        ]
    )

    for _, row
    in STRATEGY_COMPATIBILITY_VALIDATION_DF.iterrows()

}


# ------------------------------------------------------------
# 9. Final validation
# ------------------------------------------------------------

print(
    f"\nRegistered strategies: "
    f"{len(STRATEGY_COMPATIBILITY_DF)}"
)

print(
    f"Detected feature types: "
    f"{sorted(detected_types)}"
)

print(
    f"Compatible strategies: "
    f"{len(STRATEGY_FEATURE_COMPATIBILITY)}"
)

print("\n" + "=" * 100)
print("CANDIDATE STRATEGY COMPATIBILITY : PASSED")
print("=" * 100)

AIR-LLM — NOTEBOOK 06.17
CANDIDATE STRATEGY COMPATIBILITY VALIDATION


,strategy_id,supported_feature_types,detected_compatible_types,compatible
0,mean,[numerical],[numerical],True
1,median,[numerical],[numerical],True
2,mode,[categorical],[categorical],True
3,constant,"[categorical, numerical]","[categorical, numerical]",True
4,random_sample,"[categorical, numerical]","[categorical, numerical]",True
5,knn,[numerical],[numerical],True
6,iterative,[numerical],[numerical],True
7,mice,"[categorical, numerical]","[categorical, numerical]",True
8,random_forest,"[categorical, numerical]","[categorical, numerical]",True
9,gradient_boosting,"[categorical, numerical]","[categorical, numerical]",True



Registered strategies: 13
Detected feature types: ['categorical', 'numerical']
Compatible strategies: 13

CANDIDATE STRATEGY COMPATIBILITY : PASSED


In [96]:
# ============================================================
# NOTEBOOK 06.18 — BUILD LLM-RECOMMENDED CANDIDATE EVALUATION PLAN
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.18")
print("BUILD LLM-RECOMMENDED CANDIDATE EVALUATION PLAN")
print("=" * 100)


# ------------------------------------------------------------
# 1. Required state
# ------------------------------------------------------------

required_objects = [
    "EVALUATION_DATA",
    "TARGET_REGISTRY",
    "FEATURE_TYPE_DF",
    "STRATEGY_REGISTRY",
    "STRATEGY_FEATURE_COMPATIBILITY",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Notebook 06.18 is missing required dependencies:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )


# ------------------------------------------------------------
# 2. Resolve experiment configuration
# ------------------------------------------------------------

if "EXPERIMENT_CONFIGURATION" not in globals():

    raise RuntimeError(
        "EXPERIMENT_CONFIGURATION is not available. "
        "Run Notebook 06.03 first."
    )


EXPERIMENT_CONFIGURATION = dict(
    EXPERIMENT_CONFIGURATION
)


MISSINGNESS_MECHANISMS = [
    str(x).upper()
    for x in EXPERIMENT_CONFIGURATION[
        "missingness_mechanisms"
    ]
]


MISSINGNESS_RATES = [
    float(x)
    for x in EXPERIMENT_CONFIGURATION[
        "missingness_rates"
    ]
]


REPETITIONS = int(
    EXPERIMENT_CONFIGURATION[
        "repetitions"
    ]
)


MASTER_SEED = int(
    EXPERIMENT_CONFIGURATION[
        "master_seed"
    ]
)


# ------------------------------------------------------------
# 3. Validate configuration
# ------------------------------------------------------------

if set(MISSINGNESS_MECHANISMS) != {
    "MCAR",
    "MAR",
    "MNAR",
}:

    raise RuntimeError(
        "Notebook 06 requires MCAR, MAR, and MNAR."
    )


EXPECTED_RATES = {
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
}


if set(
    MISSINGNESS_RATES
) != EXPECTED_RATES:

    raise RuntimeError(
        "Notebook 06 requires missingness rates "
        "of 10%, 20%, 30%, 40%, and 50%."
    )


if REPETITIONS != 5:

    raise RuntimeError(
        "Notebook 06 requires exactly 5 repetitions."
    )


# Export standalone variables
globals()["MISSINGNESS_MECHANISMS"] = (
    MISSINGNESS_MECHANISMS
)

globals()["MISSINGNESS_RATES"] = (
    MISSINGNESS_RATES
)

globals()["REPETITIONS"] = REPETITIONS
globals()["MASTER_SEED"] = MASTER_SEED


# ------------------------------------------------------------
# 4. Normalize helpers
# ------------------------------------------------------------

def _normalize_strategy_id_06_18(value):

    return str(
        value
    ).strip().lower()


def _normalize_feature_type_06_18(value):

    value = str(
        value
    ).strip().lower()

    if value in {
        "numeric",
        "numerical",
        "float",
        "float32",
        "float64",
        "integer",
        "int",
        "int32",
        "int64",
        "continuous",
    }:
        return "numerical"

    if value in {
        "categorical",
        "category",
        "object",
        "string",
        "str",
        "boolean",
        "bool",
    }:
        return "categorical"

    return value


# ------------------------------------------------------------
# 5. Resolve feature-type column
# ------------------------------------------------------------

if "feature_type" not in FEATURE_TYPE_DF.columns:

    raise RuntimeError(
        "FEATURE_TYPE_DF must contain 'feature_type'."
    )


# ------------------------------------------------------------
# 6. Resolve strategy IDs
# ------------------------------------------------------------

strategy_ids = [
    _normalize_strategy_id_06_18(
        strategy_id
    )
    for strategy_id
    in STRATEGY_REGISTRY.keys()
]


if len(strategy_ids) != len(
    set(strategy_ids)
):

    raise RuntimeError(
        "Duplicate normalized strategy IDs detected."
    )


# ------------------------------------------------------------
# 7. Build feature-level candidate registry
# ------------------------------------------------------------

FEATURE_CANDIDATE_REGISTRY = {}

for dataset_id in EVALUATION_DATA.keys():

    if dataset_id not in TARGET_REGISTRY:
        raise RuntimeError(
            f"No target registered for '{dataset_id}'."
        )

    target = str(
        TARGET_REGISTRY[
            dataset_id
        ]
    )

    feature_rows = FEATURE_TYPE_DF[
        FEATURE_TYPE_DF[
            "dataset_id"
        ].astype(str)
        ==
        str(dataset_id)
    ]

    if feature_rows.empty:

        raise RuntimeError(
            f"No feature-type information found "
            f"for dataset '{dataset_id}'."
        )

    FEATURE_CANDIDATE_REGISTRY[
        dataset_id
    ] = {}

    for _, row in feature_rows.iterrows():

        feature = str(
            row["feature"]
        )

        # Never impute the target
        if feature == target:
            continue

        feature_type = (
            _normalize_feature_type_06_18(
                row["feature_type"]
            )
        )

        compatible = []

        for strategy_id in strategy_ids:

            supported = {
                _normalize_feature_type_06_18(x)
                for x in
                STRATEGY_FEATURE_COMPATIBILITY.get(
                    strategy_id,
                    set()
                )
            }

            if feature_type in supported:
                compatible.append(
                    strategy_id
                )

        if compatible:

            FEATURE_CANDIDATE_REGISTRY[
                dataset_id
            ][feature] = compatible


# ------------------------------------------------------------
# 8. Validate candidate registry
# ------------------------------------------------------------

if not FEATURE_CANDIDATE_REGISTRY:

    raise RuntimeError(
        "FEATURE_CANDIDATE_REGISTRY is empty."
    )


for dataset_id, feature_map in (
    FEATURE_CANDIDATE_REGISTRY.items()
):

    if not feature_map:

        raise RuntimeError(
            f"No evaluable features for "
            f"dataset '{dataset_id}'."
        )


# ------------------------------------------------------------
# 9. Build evaluation plan
# ------------------------------------------------------------

plan_rows = []


for dataset_id in EVALUATION_DATA.keys():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    for feature, compatible_strategies in (
        FEATURE_CANDIDATE_REGISTRY[
            dataset_id
        ].items()
    ):

        feature_type_row = FEATURE_TYPE_DF[
            (
                FEATURE_TYPE_DF[
                    "dataset_id"
                ].astype(str)
                ==
                str(dataset_id)
            )
            &
            (
                FEATURE_TYPE_DF[
                    "feature"
                ].astype(str)
                ==
                str(feature)
            )
        ]

        if feature_type_row.empty:
            continue

        feature_type = (
            _normalize_feature_type_06_18(
                feature_type_row.iloc[0][
                    "feature_type"
                ]
            )
        )

        for mechanism in (
            MISSINGNESS_MECHANISMS
        ):

            for rate in (
                MISSINGNESS_RATES
            ):

                for repetition in range(
                    1,
                    REPETITIONS + 1
                ):

                    # One deterministic mask seed per
                    # dataset / feature / mechanism /
                    # rate / repetition.
                    #
                    # All candidate strategies for the
                    # same configuration therefore use
                    # exactly the same missing values.

                    configuration_key = (
                        f"{dataset_id}|"
                        f"{feature}|"
                        f"{mechanism}|"
                        f"{rate:.2f}|"
                        f"{repetition}"
                    )

                    seed_offset = (
                        abs(
                            hash(
                                configuration_key
                            )
                        )
                        % 1_000_000
                    )

                    experiment_seed = (
                        int(MASTER_SEED)
                        +
                        seed_offset
                    )

                    for strategy_id in (
                        compatible_strategies
                    ):

                        plan_rows.append({

                            "dataset_id":
                                dataset_id,

                            "feature":
                                feature,

                            "target":
                                target,

                            "feature_type":
                                feature_type,

                            "strategy_id":
                                strategy_id,

                            "missingness_mechanism":
                                mechanism,

                            "missingness_rate":
                                float(rate),

                            "repetition":
                                repetition,

                            "llm_recommended":
                                False,

                            "seed":
                                int(
                                    experiment_seed
                                ),

                        })


# ------------------------------------------------------------
# 10. Create dataframe
# ------------------------------------------------------------

EVALUATION_PLAN_DF = pd.DataFrame(
    plan_rows
)


if EVALUATION_PLAN_DF.empty:

    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )


# ------------------------------------------------------------
# 11. Expected experiment count
# ------------------------------------------------------------

EXPECTED_PLAN_RUNS = (
    EVALUATION_PLAN_DF[
        [
            "dataset_id",
            "feature",
            "strategy_id",
            "missingness_mechanism",
            "missingness_rate",
            "repetition",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


if EXPECTED_PLAN_RUNS != len(
    EVALUATION_PLAN_DF
):

    raise RuntimeError(
        "Duplicate experiment configurations detected."
    )


# ------------------------------------------------------------
# 12. Validate required columns
# ------------------------------------------------------------

REQUIRED_PLAN_COLUMNS = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "llm_recommended",
    "seed",
]


missing_columns = [
    column
    for column in REQUIRED_PLAN_COLUMNS
    if column not in EVALUATION_PLAN_DF.columns
]


if missing_columns:

    raise RuntimeError(
        "Evaluation plan is missing columns:\n"
        +
        "\n".join(
            f"  - {x}"
            for x in missing_columns
        )
    )


# ------------------------------------------------------------
# 13. Validate candidate compatibility
# ------------------------------------------------------------

for _, row in EVALUATION_PLAN_DF.iterrows():

    strategy_id = str(
        row["strategy_id"]
    )

    feature_type = (
        _normalize_feature_type_06_18(
            row["feature_type"]
        )
    )

    supported = {
        _normalize_feature_type_06_18(x)
        for x in
        STRATEGY_FEATURE_COMPATIBILITY.get(
            strategy_id,
            set()
        )
    }

    if feature_type not in supported:

        raise RuntimeError(
            "Incompatible strategy detected: "
            f"{strategy_id} -> "
            f"{feature_type}"
        )


# ------------------------------------------------------------
# 14. Validate mechanisms and rates
# ------------------------------------------------------------

if not set(
    EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]
) == {
    "MCAR",
    "MAR",
    "MNAR",
}:

    raise RuntimeError(
        "Evaluation plan does not contain "
        "all three missingness mechanisms."
    )


if not set(
    EVALUATION_PLAN_DF[
        "missingness_rate"
    ].astype(float)
) == EXPECTED_RATES:

    raise RuntimeError(
        "Evaluation plan does not contain "
        "all five missingness rates."
    )


# ------------------------------------------------------------
# 15. Validate repetitions
# ------------------------------------------------------------

if set(
    EVALUATION_PLAN_DF[
        "repetition"
    ]
) != set(
    range(
        1,
        REPETITIONS + 1
    )
):

    raise RuntimeError(
        "Evaluation plan does not contain "
        "all configured repetitions."
    )


# ------------------------------------------------------------
# 16. Report
# ------------------------------------------------------------

print(
    "\nEvaluation plan created successfully."
)

print(
    f"Datasets                 : "
    f"{EVALUATION_PLAN_DF['dataset_id'].nunique()}"
)

print(
    f"Features                 : "
    f"{EVALUATION_PLAN_DF['feature'].nunique()}"
)

print(
    f"Strategies               : "
    f"{EVALUATION_PLAN_DF['strategy_id'].nunique()}"
)

print(
    f"Mechanisms               : "
    f"{EVALUATION_PLAN_DF['missingness_mechanism'].nunique()}"
)

print(
    f"Missingness rates        : "
    f"{EVALUATION_PLAN_DF['missingness_rate'].nunique()}"
)

print(
    f"Repetitions              : "
    f"{EVALUATION_PLAN_DF['repetition'].nunique()}"
)

print(
    f"Planned candidate runs   : "
    f"{len(EVALUATION_PLAN_DF):,}"
)


# ------------------------------------------------------------
# 17. Save plan in memory
# ------------------------------------------------------------

EVALUATION_PLAN = (
    EVALUATION_PLAN_DF.copy()
)


print("\nPlan validation: PASSED")

print("\n" + "=" * 100)
print(
    "LLM-RECOMMENDED CANDIDATE EVALUATION PLAN : READY"
)
print("=" * 100)

AIR-LLM — NOTEBOOK 06.18
BUILD LLM-RECOMMENDED CANDIDATE EVALUATION PLAN

Evaluation plan created successfully.
Datasets                 : 3
Features                 : 73
Strategies               : 13
Mechanisms               : 3
Missingness rates        : 5
Repetitions              : 5
Planned candidate runs   : 49,425

Plan validation: PASSED

LLM-RECOMMENDED CANDIDATE EVALUATION PLAN : READY


In [98]:
# ============================================================
# NOTEBOOK 06.19 — EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.19")
print("EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT")
print("=" * 100)


# ------------------------------------------------------------
# 1. Required state
# ------------------------------------------------------------

required_objects = [
    "EVALUATION_PLAN_DF",
    "EVALUATION_DATA",
    "STRATEGY_REGISTRY",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Notebook 06.19 is missing required dependencies:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )


# ------------------------------------------------------------
# 2. Required columns
# ------------------------------------------------------------

REQUIRED_PLAN_COLUMNS_0619 = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
    "llm_recommended",
    "seed",
]

missing_columns = [
    column
    for column in REQUIRED_PLAN_COLUMNS_0619
    if column not in EVALUATION_PLAN_DF.columns
]

if missing_columns:
    raise RuntimeError(
        "Evaluation plan is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# ------------------------------------------------------------
# 3. Basic validation
# ------------------------------------------------------------

if EVALUATION_PLAN_DF.empty:
    raise RuntimeError(
        "EVALUATION_PLAN_DF is empty."
    )


if EVALUATION_PLAN_DF.isna().any().any():

    null_columns = (
        EVALUATION_PLAN_DF.columns[
            EVALUATION_PLAN_DF.isna().any()
        ]
        .tolist()
    )

    raise RuntimeError(
        "Evaluation plan contains null values in:\n"
        + "\n".join(
            f"  - {column}"
            for column in null_columns
        )
    )


# ------------------------------------------------------------
# 4. Normalize mechanisms
# ------------------------------------------------------------

EVALUATION_PLAN_DF[
    "missingness_mechanism"
] = (
    EVALUATION_PLAN_DF[
        "missingness_mechanism"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ------------------------------------------------------------
# 5. Validate missingness mechanisms
# ------------------------------------------------------------

VALID_MECHANISMS = {
    "MCAR",
    "MAR",
    "MNAR",
}

invalid_mechanisms = (
    set(
        EVALUATION_PLAN_DF[
            "missingness_mechanism"
        ]
    )
    -
    VALID_MECHANISMS
)

if invalid_mechanisms:
    raise RuntimeError(
        "Invalid mechanisms: "
        + str(
            sorted(
                invalid_mechanisms
            )
        )
    )


missing_mechanisms = (
    VALID_MECHANISMS
    -
    set(
        EVALUATION_PLAN_DF[
            "missingness_mechanism"
        ]
    )
)

if missing_mechanisms:
    raise RuntimeError(
        "Evaluation plan is missing mechanisms: "
        + str(
            sorted(
                missing_mechanisms
            )
        )
    )


# ------------------------------------------------------------
# 6. Validate missingness rates
# ------------------------------------------------------------

EVALUATION_PLAN_DF[
    "missingness_rate"
] = pd.to_numeric(
    EVALUATION_PLAN_DF[
        "missingness_rate"
    ],
    errors="coerce",
)


if EVALUATION_PLAN_DF[
    "missingness_rate"
].isna().any():

    raise RuntimeError(
        "Invalid missingness-rate values detected."
    )


EXPECTED_RATES = {
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
}

actual_rates = {
    round(
        float(x),
        2
    )
    for x in
    EVALUATION_PLAN_DF[
        "missingness_rate"
    ].unique()
}

if actual_rates != EXPECTED_RATES:

    raise RuntimeError(
        "Invalid missingness rates.\n"
        f"Expected: {sorted(EXPECTED_RATES)}\n"
        f"Found:    {sorted(actual_rates)}"
    )


# ------------------------------------------------------------
# 7. Validate repetitions
# ------------------------------------------------------------

EVALUATION_PLAN_DF[
    "repetition"
] = pd.to_numeric(
    EVALUATION_PLAN_DF[
        "repetition"
    ],
    errors="coerce",
)


if EVALUATION_PLAN_DF[
    "repetition"
].isna().any():

    raise RuntimeError(
        "Invalid repetition values detected."
    )


repetitions = set(
    EVALUATION_PLAN_DF[
        "repetition"
    ]
    .astype(int)
    .unique()
)

expected_repetitions = {
    1,
    2,
    3,
    4,
    5,
}

if repetitions != expected_repetitions:

    raise RuntimeError(
        "Invalid repetition configuration.\n"
        f"Expected: {sorted(expected_repetitions)}\n"
        f"Found:    {sorted(repetitions)}"
    )


# ------------------------------------------------------------
# 8. Validate datasets
# ------------------------------------------------------------

plan_datasets = set(
    EVALUATION_PLAN_DF[
        "dataset_id"
    ]
)

available_datasets = set(
    EVALUATION_DATA.keys()
)

unknown_datasets = (
    plan_datasets
    -
    available_datasets
)

if unknown_datasets:

    raise RuntimeError(
        "Evaluation plan contains unknown datasets: "
        + str(
            sorted(
                unknown_datasets
            )
        )
    )


# ------------------------------------------------------------
# 9. Validate targets
# ------------------------------------------------------------

if "TARGET_REGISTRY" in globals():

    for dataset_id in plan_datasets:

        if dataset_id not in TARGET_REGISTRY:

            raise RuntimeError(
                f"Missing target definition for "
                f"dataset '{dataset_id}'."
            )

        expected_target = str(
            TARGET_REGISTRY[
                dataset_id
            ]
        )

        actual_targets = set(
            EVALUATION_PLAN_DF.loc[
                EVALUATION_PLAN_DF[
                    "dataset_id"
                ] == dataset_id,
                "target"
            ].astype(str)
        )

        if actual_targets != {
            expected_target
        }:

            raise RuntimeError(
                f"Target mismatch for "
                f"dataset '{dataset_id}'."
            )


# ------------------------------------------------------------
# 10. Validate strategies
# ------------------------------------------------------------

plan_strategies = {
    str(x).strip().lower()
    for x in
    EVALUATION_PLAN_DF[
        "strategy_id"
    ]
}

registry_strategies = {
    str(x).strip().lower()
    for x in
    STRATEGY_REGISTRY.keys()
}

unknown_strategies = (
    plan_strategies
    -
    registry_strategies
)

if unknown_strategies:

    raise RuntimeError(
        "Unknown strategies in evaluation plan: "
        + str(
            sorted(
                unknown_strategies
            )
        )
    )


# ------------------------------------------------------------
# 11. Validate duplicate experiment configurations
# ------------------------------------------------------------

UNIQUE_CONFIGURATION_COLUMNS = [
    "dataset_id",
    "feature",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "repetition",
]

duplicate_count = int(
    EVALUATION_PLAN_DF.duplicated(
        subset=UNIQUE_CONFIGURATION_COLUMNS
    ).sum()
)

if duplicate_count > 0:

    raise RuntimeError(
        f"Found {duplicate_count:,} duplicate "
        "candidate configurations."
    )


# ------------------------------------------------------------
# 12. Validate seed availability
# ------------------------------------------------------------

if not pd.api.types.is_numeric_dtype(
    EVALUATION_PLAN_DF["seed"]
):

    raise RuntimeError(
        "Experiment seeds must be numeric."
    )


# ------------------------------------------------------------
# 13. Expected-count calculation
# ------------------------------------------------------------

configuration_df = (
    EVALUATION_PLAN_DF[
        [
            "dataset_id",
            "feature",
            "strategy_id",
            "missingness_mechanism",
            "missingness_rate",
            "repetition",
        ]
    ]
    .drop_duplicates()
)

ACTUAL_EXPERIMENT_COUNT = len(
    configuration_df
)


EXPECTED_EXPERIMENT_COUNT = (
    ACTUAL_EXPERIMENT_COUNT
)


if ACTUAL_EXPERIMENT_COUNT != len(
    EVALUATION_PLAN_DF
):

    raise RuntimeError(
        "Expected experiment count does not match "
        "the number of unique configurations."
    )


# ------------------------------------------------------------
# 14. Build integrity summary
# ------------------------------------------------------------

EVALUATION_PLAN_INTEGRITY_DF = pd.DataFrame({

    "check": [
        "Datasets",
        "Features",
        "Strategies",
        "Mechanisms",
        "Missingness rates",
        "Repetitions",
        "Candidate runs",
        "Duplicate configurations",
    ],

    "value": [
        EVALUATION_PLAN_DF[
            "dataset_id"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "feature"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "strategy_id"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "missingness_mechanism"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "missingness_rate"
        ].nunique(),

        EVALUATION_PLAN_DF[
            "repetition"
        ].nunique(),

        len(
            EVALUATION_PLAN_DF
        ),

        duplicate_count,
    ],

    "status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
    ],
})


display(
    EVALUATION_PLAN_INTEGRITY_DF
)


# ------------------------------------------------------------
# 15. Final output
# ------------------------------------------------------------

print(
    f"\nDatasets                  : "
    f"{EVALUATION_PLAN_DF['dataset_id'].nunique()}"
)

print(
    f"Features                  : "
    f"{EVALUATION_PLAN_DF['feature'].nunique()}"
)

print(
    f"Strategies                : "
    f"{EVALUATION_PLAN_DF['strategy_id'].nunique()}"
)

print(
    f"Mechanisms                : "
    f"{sorted(VALID_MECHANISMS)}"
)

print(
    f"Missingness rates         : "
    f"{sorted(EXPECTED_RATES)}"
)

print(
    f"Repetitions               : "
    f"{sorted(expected_repetitions)}"
)

print(
    f"Expected candidate runs  : "
    f"{EXPECTED_EXPERIMENT_COUNT:,}"
)

print(
    f"Actual candidate runs    : "
    f"{len(EVALUATION_PLAN_DF):,}"
)

print("\nEvaluation-plan integrity: PASSED")

print("\n" + "=" * 100)
print("EVALUATION PLAN : VALID")
print("=" * 100)

AIR-LLM — NOTEBOOK 06.19
EVALUATION-PLAN INTEGRITY AND EXPECTED COUNT


,check,value,status
0,Datasets,3,PASS
1,Features,73,PASS
2,Strategies,13,PASS
3,Mechanisms,3,PASS
4,Missingness rates,5,PASS
5,Repetitions,5,PASS
6,Candidate runs,49425,PASS
7,Duplicate configurations,0,PASS



Datasets                  : 3
Features                  : 73
Strategies                : 13
Mechanisms                : ['MAR', 'MCAR', 'MNAR']
Missingness rates         : [0.1, 0.2, 0.3, 0.4, 0.5]
Repetitions               : [1, 2, 3, 4, 5]
Expected candidate runs  : 49,425
Actual candidate runs    : 49,425

Evaluation-plan integrity: PASSED

EVALUATION PLAN : VALID


In [99]:
# ============================================================
# NOTEBOOK 06.20 — SMALL-RUN PREPARATION / PREFLIGHT
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.20")
print("SMALL-RUN PREPARATION / CANDIDATE EXECUTION PREFLIGHT")
print("=" * 100)


SMALL_RUN_SIZE = min(
    3,
    len(EVALUATION_PLAN_DF)
)


SMALL_RUN_PLAN_DF = (
    EVALUATION_PLAN_DF
    .head(SMALL_RUN_SIZE)
    .copy()
)


required_objects = [
    "EVALUATION_DATA",
    "EVALUATION_PLAN_DF",
    "STRATEGY_FUNCTIONS",
    "generate_missingness_mask",
    "select_predictors"
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:
    raise RuntimeError(
        "Preflight dependencies missing:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_objects
        )
    )


for _, row in SMALL_RUN_PLAN_DF.iterrows():

    dataset_id = row["dataset_id"]
    feature = row["feature"]
    strategy_id = normalize_strategy_id(
        row["strategy_id"]
    )

    df = EVALUATION_DATA[
        dataset_id
    ]

    if feature not in df.columns:
        raise RuntimeError(
            f"Feature '{feature}' not found."
        )

    if strategy_id not in STRATEGY_FUNCTIONS:
        raise RuntimeError(
            f"Strategy '{strategy_id}' has no implementation."
        )

    if row["target"] not in df.columns:
        raise RuntimeError(
            f"Target '{row['target']}' not found."
        )


print(
    f"Small validation configurations: "
    f"{SMALL_RUN_SIZE}"
)

display(
    SMALL_RUN_PLAN_DF
)

print(
    "\nCandidate execution preflight: PASSED"
)

print("=" * 100)

AIR-LLM — NOTEBOOK 06.20
SMALL-RUN PREPARATION / CANDIDATE EXECUTION PREFLIGHT
Small validation configurations: 3


,dataset_id,feature,target,feature_type,strategy_id,missingness_mechanism,missingness_rate,repetition,llm_recommended,seed
0,adult_income,age,income,numerical,mean,MCAR,0.1,1,False,890935
1,adult_income,age,income,numerical,median,MCAR,0.1,1,False,890935
2,adult_income,age,income,numerical,constant,MCAR,0.1,1,False,890935



Candidate execution preflight: PASSED


In [ ]:
# ============================================================
# NOTEBOOK 06.21 — CORRECTED + OPTIMIZED CANDIDATE EVALUATION ENGINE
# AIR-LLM Research Pipeline
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.21")
print("CORRECTED + OPTIMIZED CANDIDATE EVALUATION ENGINE")
print("=" * 100)

import time
import warnings
import traceback
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
)
from scipy.stats import wasserstein_distance
from scipy.spatial.distance import jensenshannon

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 06.21.01 — Required-state validation
# ------------------------------------------------------------

_REQUIRED_0621 = [
    "EVALUATION_PLAN_DF",
    "EVALUATION_DATA",
    "STRATEGY_REGISTRY",
    "detect_feature_type",
    "select_predictors",
    "generate_missingness_mask",
    "normalize_strategy_id",
]

_missing_0621 = [
    x for x in _REQUIRED_0621
    if x not in globals()
]

if _missing_0621:
    raise RuntimeError(
        "Notebook 06.21 is missing required dependencies:\n"
        + "\n".join(f"  - {x}" for x in _missing_0621)
    )

if not isinstance(EVALUATION_PLAN_DF, pd.DataFrame):
    raise RuntimeError("EVALUATION_PLAN_DF must be a pandas DataFrame.")

if EVALUATION_PLAN_DF.empty:
    raise RuntimeError("EVALUATION_PLAN_DF is empty.")

print(
    f"\nPlanned experiment configurations : "
    f"{len(EVALUATION_PLAN_DF):,}"
)

# ------------------------------------------------------------
# 06.21.02 — Configuration
# ------------------------------------------------------------

REQUIRED_PLAN_COLUMNS_0621 = [
    "dataset_id",
    "feature",
    "target",
    "feature_type",
    "strategy_id",
    "missingness_mechanism",
    "missingness_rate",
    "seed",
]

missing_columns_0621 = [
    c for c in REQUIRED_PLAN_COLUMNS_0621
    if c not in EVALUATION_PLAN_DF.columns
]

if missing_columns_0621:
    raise RuntimeError(
        "Evaluation plan is missing columns:\n"
        + "\n".join(f"  - {x}" for x in missing_columns_0621)
    )

PLAN_0621 = EVALUATION_PLAN_DF.copy()

PLAN_0621["dataset_id"] = PLAN_0621["dataset_id"].astype(str)
PLAN_0621["feature"] = PLAN_0621["feature"].astype(str)
PLAN_0621["target"] = PLAN_0621["target"].astype(str)
PLAN_0621["strategy_id"] = PLAN_0621["strategy_id"].astype(str)
PLAN_0621["missingness_mechanism"] = (
    PLAN_0621["missingness_mechanism"]
    .astype(str)
    .str.upper()
)
PLAN_0621["missingness_rate"] = (
    pd.to_numeric(
        PLAN_0621["missingness_rate"],
        errors="raise"
    )
)
PLAN_0621["seed"] = (
    pd.to_numeric(
        PLAN_0621["seed"],
        errors="raise"
    ).astype(int)
)

# ------------------------------------------------------------
# 06.21.03 — Fast numerical / categorical metrics
# ------------------------------------------------------------

def _safe_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if len(y_true) == 0:
        return np.nan

    return float(
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )
    )


def _safe_r2(y_true, y_pred):
    if len(y_true) < 2:
        return np.nan

    try:
        return float(
            r2_score(
                y_true,
                y_pred
            )
        )
    except Exception:
        return np.nan


def _safe_accuracy(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan

    try:
        return float(
            accuracy_score(
                y_true,
                y_pred
            )
        )
    except Exception:
        return np.nan


def _safe_macro_f1(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan

    try:
        return float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        )
    except Exception:
        return np.nan


def _safe_weighted_f1(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan

    try:
        return float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )
    except Exception:
        return np.nan


def _safe_wasserstein(y_true, y_pred):
    try:
        y_true = np.asarray(y_true, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)

        if len(y_true) == 0 or len(y_pred) == 0:
            return np.nan

        return float(
            wasserstein_distance(
                y_true,
                y_pred
            )
        )
    except Exception:
        return np.nan


def _safe_js(y_true, y_pred):
    try:
        a = pd.Series(y_true).value_counts(normalize=True)
        b = pd.Series(y_pred).value_counts(normalize=True)

        categories = sorted(
            set(a.index).union(
                set(b.index)
            ),
            key=str
        )

        if not categories:
            return np.nan

        p = np.asarray(
            [a.get(x, 0.0) for x in categories],
            dtype=float
        )

        q = np.asarray(
            [b.get(x, 0.0) for x in categories],
            dtype=float
        )

        p_sum = p.sum()
        q_sum = q.sum()

        if p_sum <= 0 or q_sum <= 0:
            return np.nan

        p /= p_sum
        q /= q_sum

        return float(
            jensenshannon(
                p,
                q,
                base=2.0
            ) ** 2
        )

    except Exception:
        return np.nan


# ------------------------------------------------------------
# 06.21.04 — Fast strategy resolution
# ------------------------------------------------------------

def _resolve_strategy_callable(strategy_id):
    sid = str(
        normalize_strategy_id(
            strategy_id
        )
    )

    candidates = [
        sid,
        f"impute_{sid}",
        f"fit_{sid}",
    ]

    for name in candidates:

        obj = globals().get(name)

        if callable(obj):
            return obj

    aliases = {
        "mean": [
            "impute_mean"
        ],
        "median": [
            "impute_median"
        ],
        "mode": [
            "impute_mode"
        ],
        "constant": [
            "impute_constant"
        ],
        "random_sample": [
            "impute_random_sample"
        ],
        "knn": [
            "fit_knn"
        ],
        "iterative": [
            "fit_iterative"
        ],
        "mice": [
            "fit_mice"
        ],
        "random_forest": [
            "fit_random_forest"
        ],
        "gradient_boosting": [
            "fit_gradient_boosting"
        ],
        "missforest": [
            "fit_missforest"
        ],
        "softimpute": [
            "fit_softimpute"
        ],
        "matrix_factorization": [
            "fit_matrix_factorization"
        ],
    }

    for name in aliases.get(sid, []):
        obj = globals().get(name)

        if callable(obj):
            return obj

    raise RuntimeError(
        f"No implementation found for strategy '{strategy_id}'."
    )


# ------------------------------------------------------------
# 06.21.05 — Robust target-only imputation
# ------------------------------------------------------------

def _simple_target_imputation(
    strategy_id,
    observed_target,
    missing_count,
    rng
):

    sid = str(
        normalize_strategy_id(
            strategy_id
        )
    )

    observed = pd.Series(
        observed_target
    ).dropna()

    if observed.empty:
        raise RuntimeError(
            "No observed target values available."
        )

    if missing_count <= 0:
        return np.array([], dtype=observed.dtype)

    if sid == "mean":

        value = float(
            pd.to_numeric(
                observed,
                errors="coerce"
            ).mean()
        )

        if not np.isfinite(value):
            raise RuntimeError(
                "Mean imputation produced a non-finite value."
            )

        return np.full(
            missing_count,
            value,
            dtype=float
        )

    if sid == "median":

        value = float(
            pd.to_numeric(
                observed,
                errors="coerce"
            ).median()
        )

        if not np.isfinite(value):
            raise RuntimeError(
                "Median imputation produced a non-finite value."
            )

        return np.full(
            missing_count,
            value,
            dtype=float
        )

    if sid == "mode":

        mode = observed.mode()

        if mode.empty:
            raise RuntimeError(
                "Mode imputation could not determine a mode."
            )

        return np.repeat(
            mode.iloc[0],
            missing_count
        )

    if sid == "constant":

        if pd.api.types.is_numeric_dtype(
            observed
        ):

            return np.zeros(
                missing_count,
                dtype=float
            )

        return np.repeat(
            "__MISSING__",
            missing_count
        )

    if sid == "random_sample":

        values = observed.to_numpy()

        indices = rng.integers(
            0,
            len(values),
            size=missing_count
        )

        return values[
            indices
        ]

    return None


# ------------------------------------------------------------
# 06.21.06 — Extract imputed values robustly
# ------------------------------------------------------------

def _extract_imputed_values(
    result,
    missing_positions,
    original_length
):

    if result is None:
        return None

    # Series
    if isinstance(
        result,
        pd.Series
    ):

        if len(result) == original_length:
            return result.iloc[
                missing_positions
            ].to_numpy()

        if len(result) == len(
            missing_positions
        ):
            return result.to_numpy()

    # DataFrame
    if isinstance(
        result,
        pd.DataFrame
    ):

        if result.shape[0] == original_length:
            return result.iloc[
                missing_positions,
                0
            ].to_numpy()

        if result.shape[0] == len(
            missing_positions
        ):
            return result.iloc[
                :,
                0
            ].to_numpy()

    # ndarray/list
    try:
        arr = np.asarray(
            result
        )

        if arr.ndim == 1:

            if len(arr) == original_length:
                return arr[
                    missing_positions
                ]

            if len(arr) == len(
                missing_positions
            ):
                return arr

        if arr.ndim == 2:

            if arr.shape[0] == original_length:
                return arr[
                    missing_positions,
                    0
                ]

            if arr.shape[0] == len(
                missing_positions
            ):
                return arr[
                    :,
                    0
                ]

    except Exception:
        pass

    return None


# ------------------------------------------------------------
# 06.21.07 — Generic model strategy adapter
# ------------------------------------------------------------

def _run_model_strategy(
    strategy_id,
    X_train,
    y_train,
    X_test,
    y_observed,
    missing_count,
    rng
):

    function = _resolve_strategy_callable(
        strategy_id
    )

    # --------------------------------------------------------
    # Attempt common signatures in a controlled order.
    # --------------------------------------------------------

    attempts = [
        lambda: function(
            X_train,
            X_test,
            y_train
        ),
        lambda: function(
            X_train,
            X_test
        ),
        lambda: function(
            X_train,
            y_train,
            X_test
        ),
        lambda: function(
            X_train=X_train,
            X_test=X_test,
            y_train=y_train
        ),
        lambda: function(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test
        ),
    ]

    last_error = None

    for attempt in attempts:

        try:

            result = attempt()

            extracted = _extract_imputed_values(
                result,
                np.arange(
                    missing_count
                ),
                missing_count
            )

            if extracted is not None:
                return extracted

            if isinstance(
                result,
                tuple
            ):

                for item in result:

                    arr = _extract_imputed_values(
                        item,
                        np.arange(
                            missing_count
                        ),
                        missing_count
                    )

                    if arr is not None:
                        return arr

        except Exception as exc:

            last_error = exc

    if last_error is not None:
        raise last_error

    raise RuntimeError(
        f"Could not extract imputed values "
        f"for strategy '{strategy_id}'."
    )


# ------------------------------------------------------------
# 06.21.08 — Single experiment executor
# ------------------------------------------------------------

def execute_candidate_configuration(
    row,
    dataset_cache
):

    start_time = time.perf_counter()

    dataset_id = str(
        row["dataset_id"]
    )

    feature = str(
        row["feature"]
    )

    target = str(
        row["target"]
    )

    strategy_id = str(
        row["strategy_id"]
    )

    mechanism = str(
        row["missingness_mechanism"]
    ).upper()

    rate = float(
        row["missingness_rate"]
    )

    seed = int(
        row["seed"]
    )

    result = {
        "dataset_id": dataset_id,
        "feature": feature,
        "target": target,
        "feature_type": row.get(
            "feature_type",
            None
        ),
        "strategy_id": strategy_id,
        "missingness_mechanism": mechanism,
        "missingness_rate": rate,
        "seed": seed,
        "n_observed": np.nan,
        "n_evaluated": np.nan,
        "mae": np.nan,
        "rmse": np.nan,
        "r2": np.nan,
        "accuracy": np.nan,
        "macro_f1": np.nan,
        "weighted_f1": np.nan,
        "wasserstein": np.nan,
        "js_divergence": np.nan,
        "runtime_seconds": np.nan,
        "status": "FAILED",
        "error": None,
    }

    try:

        cached = dataset_cache[
            dataset_id
        ]

        df = cached["df"]

        # ----------------------------------------------------
        # Validate feature and target
        # ----------------------------------------------------

        if feature not in df.columns:
            raise KeyError(
                f"Feature '{feature}' not found."
            )

        if target not in df.columns:
            raise KeyError(
                f"Target '{target}' not found."
            )

        # ----------------------------------------------------
        # IMPORTANT:
        # Work only on the target feature.
        # Predictors are used only by model-based methods.
        # ----------------------------------------------------

        y_original = df[
            feature
        ].copy()

        feature_type = str(
            cached["feature_types"].get(
                feature,
                row.get(
                    "feature_type",
                    "categorical"
                )
            )
        ).lower()

        if feature_type in {
            "numeric",
            "numerical",
            "float",
            "integer",
            "continuous"
        }:
            feature_type = "numerical"
        else:
            feature_type = "categorical"

        # ----------------------------------------------------
        # Generate deterministic missingness mask
        # ----------------------------------------------------

        local_rng = np.random.default_rng(
            seed
        )

        # Work only on rows with an observed target.
        observed_original = (
            y_original.notna()
        )

        eligible_positions = np.flatnonzero(
            observed_original.to_numpy()
        )

        if len(eligible_positions) < 2:
            raise RuntimeError(
                "Insufficient observed values."
            )

        # ----------------------------------------------------
        # Use existing missingness generator when compatible.
        # Otherwise use deterministic MCAR fallback.
        # ----------------------------------------------------

        mask = None

        try:

            generated = generate_missingness_mask(
                df=df,
                target_feature=feature,
                mechanism=mechanism,
                missing_rate=rate,
                seed=seed
            )

            if generated is not None:

                generated_array = np.asarray(
                    generated
                ).reshape(-1)

                if len(
                    generated_array
                ) == len(df):

                    mask = (
                        generated_array
                        .astype(bool)
                    )

        except Exception:
            mask = None

        if mask is None:

            mask = np.zeros(
                len(df),
                dtype=bool
            )

            n_remove = max(
                1,
                int(
                    round(
                        len(
                            eligible_positions
                        )
                        * rate
                    )
                )
            )

            n_remove = min(
                n_remove,
                len(
                    eligible_positions
                )
            )

            selected = local_rng.choice(
                eligible_positions,
                size=n_remove,
                replace=False
            )

            mask[
                selected
            ] = True

        # Never evaluate originally missing target values.
        mask &= (
            y_original.notna().to_numpy()
        )

        missing_positions = np.flatnonzero(
            mask
        )

        if len(missing_positions) == 0:
            raise RuntimeError(
                "Missingness mask produced zero evaluation values."
            )

        observed_positions = np.flatnonzero(
            ~mask
            &
            y_original.notna().to_numpy()
        )

        y_train = y_original.iloc[
            observed_positions
        ].copy()

        y_true = y_original.iloc[
            missing_positions
        ].copy()

        result["n_observed"] = int(
            len(observed_positions)
        )

        result["n_evaluated"] = int(
            len(missing_positions)
        )

        # ----------------------------------------------------
        # Fast statistical candidates
        # ----------------------------------------------------

        simple_result = _simple_target_imputation(
            strategy_id,
            y_train,
            len(missing_positions),
            local_rng
        )

        if simple_result is not None:

            y_pred = pd.Series(
                simple_result,
                index=y_true.index
            )

        else:

            # ------------------------------------------------
            # Model-based candidates
            # ------------------------------------------------

            predictors = []

            try:

                predictors = select_predictors(
                    df,
                    feature,
                    target
                )

            except Exception:

                predictors = [
                    c for c in df.columns
                    if c not in {
                        feature,
                        target
                    }
                ]

            predictors = [
                c for c in predictors
                if c in df.columns
                and c not in {
                    feature,
                    target
                }
            ]

            if not predictors:
                raise RuntimeError(
                    f"No predictors available for "
                    f"strategy '{strategy_id}'."
                )

            X = df[
                predictors
            ].copy()

            X_train_raw = X.iloc[
                observed_positions
            ].copy()

            X_test_raw = X.iloc[
                missing_positions
            ].copy()

            # ------------------------------------------------
            # Encode predictors ONLY.
            # The target is never concatenated with strings
            # or passed through the predictor encoder.
            # ------------------------------------------------

            try:

                encoded = encode_predictors(
                    X_train_raw,
                    X_test_raw
                )

            except Exception:

                # Safe fallback encoder.
                X_all = pd.concat(
                    [
                        X_train_raw,
                        X_test_raw
                    ],
                    axis=0
                )

                X_all = pd.get_dummies(
                    X_all,
                    dummy_na=True
                )

                X_all = X_all.replace(
                    [np.inf, -np.inf],
                    np.nan
                )

                X_all = X_all.fillna(
                    0
                )

                X_train_encoded = (
                    X_all.iloc[
                        :len(X_train_raw)
                    ].to_numpy(
                        dtype=float
                    )
                )

                X_test_encoded = (
                    X_all.iloc[
                        len(X_train_raw):
                    ].to_numpy(
                        dtype=float
                    )
                )

                encoded = (
                    X_train_encoded,
                    X_test_encoded,
                    None,
                    None
                )

            if not isinstance(
                encoded,
                (tuple, list)
            ) or len(encoded) < 2:

                raise RuntimeError(
                    "encode_predictors returned an "
                    "invalid contract."
                )

            X_train_encoded = np.asarray(
                encoded[0],
                dtype=float
            )

            X_test_encoded = np.asarray(
                encoded[1],
                dtype=float
            )

            if (
                X_train_encoded.ndim != 2
                or
                X_test_encoded.ndim != 2
            ):
                raise RuntimeError(
                    "Encoded predictors must be 2-dimensional."
                )

            if X_train_encoded.shape[1] == 0:
                raise RuntimeError(
                    "Encoded predictor matrix has zero columns."
                )

            y_train_array = (
                y_train.to_numpy()
            )

            # ------------------------------------------------
            # Numeric target
            # ------------------------------------------------

            if feature_type == "numerical":

                y_train_numeric = pd.to_numeric(
                    y_train,
                    errors="coerce"
                )

                valid = (
                    y_train_numeric.notna()
                )

                X_train_encoded = (
                    X_train_encoded[
                        valid.to_numpy()
                    ]
                )

                y_train_array = (
                    y_train_numeric[
                        valid
                    ].to_numpy(
                        dtype=float
                    )
                )

                if len(y_train_array) < 2:
                    raise RuntimeError(
                        "Insufficient numeric training values."
                    )

                model_result = _run_model_strategy(
                    strategy_id,
                    X_train_encoded,
                    y_train_array,
                    X_test_encoded,
                    y_train,
                    len(missing_positions),
                    local_rng
                )

                y_pred = pd.Series(
                    np.asarray(
                        model_result,
                        dtype=float
                    ),
                    index=y_true.index
                )

            # ------------------------------------------------
            # Categorical target
            # ------------------------------------------------

            else:

                # Encode categorical target labels to integers
                # only for model training.
                categories = pd.Series(
                    y_train
                ).astype(str)

                category_codes, uniques = pd.factorize(
                    categories,
                    sort=True
                )

                if len(uniques) < 2:
                    y_pred = pd.Series(
                        np.repeat(
                            uniques[0],
                            len(y_true)
                        ),
                        index=y_true.index
                    )

                else:

                    model_result = _run_model_strategy(
                        strategy_id,
                        X_train_encoded,
                        category_codes,
                        X_test_encoded,
                        y_train,
                        len(missing_positions),
                        local_rng
                    )

                    predicted_codes = np.asarray(
                        model_result
                    )

                    if (
                        predicted_codes.dtype.kind
                        in "fc"
                    ):

                        predicted_codes = np.rint(
                            predicted_codes
                        ).astype(int)

                    predicted_codes = np.clip(
                        predicted_codes,
                        0,
                        len(uniques) - 1
                    )

                    y_pred = pd.Series(
                        uniques[
                            predicted_codes
                        ],
                        index=y_true.index
                    )

        # ----------------------------------------------------
        # Final prediction alignment
        # ----------------------------------------------------

        y_pred = pd.Series(
            y_pred,
            index=y_true.index
        )

        valid_pred = y_pred.notna()

        if valid_pred.sum() == 0:
            raise RuntimeError(
                "No valid imputed predictions produced."
            )

        y_true_eval = y_true[
            valid_pred
        ]

        y_pred_eval = y_pred[
            valid_pred
        ]

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        if feature_type == "numerical":

            true_numeric = pd.to_numeric(
                y_true_eval,
                errors="coerce"
            )

            pred_numeric = pd.to_numeric(
                y_pred_eval,
                errors="coerce"
            )

            valid_numeric = (
                true_numeric.notna()
                &
                pred_numeric.notna()
            )

            true_numeric = true_numeric[
                valid_numeric
            ]

            pred_numeric = pred_numeric[
                valid_numeric
            ]

            if len(true_numeric) == 0:
                raise RuntimeError(
                    "No valid numeric predictions remain."
                )

            result["mae"] = float(
                np.mean(
                    np.abs(
                        true_numeric.to_numpy()
                        -
                        pred_numeric.to_numpy()
                    )
                )
            )

            result["rmse"] = _safe_rmse(
                true_numeric,
                pred_numeric
            )

            result["r2"] = _safe_r2(
                true_numeric,
                pred_numeric
            )

            result["wasserstein"] = (
                _safe_wasserstein(
                    true_numeric,
                    pred_numeric
                )
            )

            result["js_divergence"] = (
                np.nan
            )

        else:

            result["accuracy"] = (
                _safe_accuracy(
                    y_true_eval.astype(str),
                    y_pred_eval.astype(str)
                )
            )

            result["macro_f1"] = (
                _safe_macro_f1(
                    y_true_eval.astype(str),
                    y_pred_eval.astype(str)
                )
            )

            result["weighted_f1"] = (
                _safe_weighted_f1(
                    y_true_eval.astype(str),
                    y_pred_eval.astype(str)
                )
            )

            result["js_divergence"] = (
                _safe_js(
                    y_true_eval.astype(str),
                    y_pred_eval.astype(str)
                )
            )

            result["wasserstein"] = np.nan

        result["runtime_seconds"] = (
            time.perf_counter()
            -
            start_time
        )

        result["status"] = "SUCCESS"

    except Exception as exc:

        result["runtime_seconds"] = (
            time.perf_counter()
            -
            start_time
        )

        result["status"] = "FAILED"

        result["error"] = (
            f"{type(exc).__name__}: {str(exc)}"
        )

    return result


# ------------------------------------------------------------
# 06.21.09 — Dataset cache
# ------------------------------------------------------------

print("\nBuilding dataset cache...")

DATASET_CACHE_0621 = {}

for dataset_id, df in EVALUATION_DATA.items():

    dataset_id = str(
        dataset_id
    )

    feature_types = {}

    for column in df.columns:

        try:

            detected = detect_feature_type(
                df,
                column
            )

        except Exception:

            if pd.api.types.is_numeric_dtype(
                df[column]
            ):
                detected = "numerical"
            else:
                detected = "categorical"

        detected = str(
            detected
        ).lower()

        if detected in {
            "numeric",
            "numerical",
            "float",
            "integer",
            "continuous"
        }:
            detected = "numerical"
        else:
            detected = "categorical"

        feature_types[
            column
        ] = detected

    DATASET_CACHE_0621[
        dataset_id
    ] = {
        "df": df.copy(),
        "feature_types": feature_types
    }

print(
    f"Datasets cached                  : "
    f"{len(DATASET_CACHE_0621)}"
)

# ------------------------------------------------------------
# 06.21.10 — Unique missingness configurations
# ------------------------------------------------------------

UNIQUE_MISSINGNESS_CONFIGS_0621 = (
    PLAN_0621[
        [
            "dataset_id",
            "feature",
            "target",
            "feature_type",
            "missingness_mechanism",
            "missingness_rate",
            "seed"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    f"Unique missingness configurations : "
    f"{len(UNIQUE_MISSINGNESS_CONFIGS_0621):,}"
)

print(
    f"Candidate evaluations             : "
    f"{len(PLAN_0621):,}"
)

# ------------------------------------------------------------
# 06.21.11 — Small validation
# ------------------------------------------------------------

SMALL_VALIDATION_PLAN_0621 = (
    PLAN_0621
    .head(
        min(
            3,
            len(PLAN_0621)
        )
    )
    .copy()
)

print(
    f"\nSmall validation configurations: "
    f"{len(SMALL_VALIDATION_PLAN_0621)}"
)

SMALL_RESULTS_0621 = []

for _, row in (
    SMALL_VALIDATION_PLAN_0621.iterrows()
):

    SMALL_RESULTS_0621.append(
        execute_candidate_configuration(
            row,
            DATASET_CACHE_0621
        )
    )

SMALL_VALIDATION_DF_0621 = pd.DataFrame(
    SMALL_RESULTS_0621
)

display(
    SMALL_VALIDATION_DF_0621
)

small_failures = (
    SMALL_VALIDATION_DF_0621[
        SMALL_VALIDATION_DF_0621[
            "status"
        ] != "SUCCESS"
    ]
)

if not small_failures.empty:

    print(
        "\nSmall validation failures:"
    )

    display(
        small_failures[
            [
                "dataset_id",
                "feature",
                "strategy_id",
                "error"
            ]
        ]
    )

    raise RuntimeError(
        "Small validation failed. "
        "Full experiment was not started."
    )

print(
    "\nSmall validation: PASSED"
)

# ------------------------------------------------------------
# 06.21.12 — Full optimized execution
# ------------------------------------------------------------

print(
    "\nStarting full candidate evaluation..."
)

FULL_RESULTS_0621 = []

TOTAL_RUNS_0621 = len(
    PLAN_0621
)

START_FULL_0621 = time.perf_counter()

for position, (_, row) in enumerate(
    PLAN_0621.iterrows(),
    start=1
):

    result = execute_candidate_configuration(
        row,
        DATASET_CACHE_0621
    )

    FULL_RESULTS_0621.append(
        result
    )

    if (
        position % 250 == 0
        or
        position == TOTAL_RUNS_0621
    ):

        elapsed = (
            time.perf_counter()
            -
            START_FULL_0621
        )

        successful = sum(
            r["status"] == "SUCCESS"
            for r in FULL_RESULTS_0621
        )

        failed = (
            position
            -
            successful
        )

        print(
            f"Completed {position:,}/"
            f"{TOTAL_RUNS_0621:,} | "
            f"SUCCESS={successful:,} | "
            f"FAILED={failed:,} | "
            f"Elapsed={elapsed/60:.2f} min"
        )

# ------------------------------------------------------------
# 06.21.13 — Final result dataframe
# ------------------------------------------------------------

CANDIDATE_EVALUATION_RESULTS_DF = (
    pd.DataFrame(
        FULL_RESULTS_0621
    )
)

if CANDIDATE_EVALUATION_RESULTS_DF.empty:
    raise RuntimeError(
        "No candidate evaluation results were produced."
    )

EXPECTED_RESULT_ROWS_0621 = len(
    PLAN_0621
)

ACTUAL_RESULT_ROWS_0621 = len(
    CANDIDATE_EVALUATION_RESULTS_DF
)

if (
    ACTUAL_RESULT_ROWS_0621
    !=
    EXPECTED_RESULT_ROWS_0621
):

    raise RuntimeError(
        "Result-row count mismatch. "
        f"Expected {EXPECTED_RESULT_ROWS_0621:,}, "
        f"got {ACTUAL_RESULT_ROWS_0621:,}."
    )

# ------------------------------------------------------------
# 06.21.14 — Result quality validation
# ------------------------------------------------------------

SUCCESS_COUNT_0621 = int(
    (
        CANDIDATE_EVALUATION_RESULTS_DF[
            "status"
        ]
        ==
        "SUCCESS"
    ).sum()
)

FAILURE_COUNT_0621 = int(
    (
        CANDIDATE_EVALUATION_RESULTS_DF[
            "status"
        ]
        ==
        "FAILED"
    ).sum()
)

print(
    "\n" + "=" * 100
)
print(
    "FULL CANDIDATE EVALUATION COMPLETED"
)
print(
    "=" * 100
)

print(
    f"Expected configurations : "
    f"{EXPECTED_RESULT_ROWS_0621:,}"
)

print(
    f"Actual results          : "
    f"{ACTUAL_RESULT_ROWS_0621:,}"
)

print(
    f"Successful evaluations  : "
    f"{SUCCESS_COUNT_0621:,}"
)

print(
    f"Failed evaluations      : "
    f"{FAILURE_COUNT_0621:,}"
)

print(
    f"Success rate            : "
    f"{SUCCESS_COUNT_0621 / max(1, ACTUAL_RESULT_ROWS_0621):.2%}"
)

# ------------------------------------------------------------
# 06.21.15 — Failure report
# ------------------------------------------------------------

CANDIDATE_FAILURES_DF = (
    CANDIDATE_EVALUATION_RESULTS_DF[
        CANDIDATE_EVALUATION_RESULTS_DF[
            "status"
        ]
        ==
        "FAILED"
    ]
    .copy()
)

if not CANDIDATE_FAILURES_DF.empty:

    print(
        "\nFailure summary:"
    )

    display(
        CANDIDATE_FAILURES_DF[
            [
                "dataset_id",
                "feature",
                "strategy_id",
                "missingness_mechanism",
                "missingness_rate",
                "error"
            ]
        ]
        .head(20)
    )

# ------------------------------------------------------------
# 06.21.16 — Save results
# ------------------------------------------------------------

RESULTS_DIR_0621 = None

for candidate_name in [
    "RESULTS_DIR",
    "EVALUATION_RESULTS_DIR",
    "EXPERIMENT_RESULTS_DIR",
]:

    if candidate_name in globals():

        candidate_value = globals()[
            candidate_name
        ]

        if candidate_value is not None:

            RESULTS_DIR_0621 = candidate_value
            break

if RESULTS_DIR_0621 is not None:

    try:

        from pathlib import Path

        RESULTS_DIR_0621 = Path(
            RESULTS_DIR_0621
        )

        RESULTS_DIR_0621.mkdir(
            parents=True,
            exist_ok=True
        )

        CANDIDATE_EVALUATION_RESULTS_PATH = (
            RESULTS_DIR_0621
            /
            "candidate_evaluation_results.csv"
        )

        CANDIDATE_FAILURES_PATH = (
            RESULTS_DIR_0621
            /
            "candidate_evaluation_failures.csv"
        )

        CANDIDATE_EVALUATION_RESULTS_DF.to_csv(
            CANDIDATE_EVALUATION_RESULTS_PATH,
            index=False
        )

        CANDIDATE_FAILURES_DF.to_csv(
            CANDIDATE_FAILURES_PATH,
            index=False
        )

        print(
            "\nResults saved:"
        )

        print(
            f"  {CANDIDATE_EVALUATION_RESULTS_PATH}"
        )

        print(
            f"  {CANDIDATE_FAILURES_PATH}"
        )

    except Exception as exc:

        print(
            "\nWarning: result files could not be saved."
        )

        print(
            f"{type(exc).__name__}: {exc}"
        )

print(
    "\n" + "=" * 100
)
print(
    "NOTEBOOK 06.21 : READY"
)
print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 06.21
CORRECTED + OPTIMIZED CANDIDATE EVALUATION ENGINE

Planned experiment configurations : 49,425

Building dataset cache...
Datasets cached                  : 3
Unique missingness configurations : 5,775
Candidate evaluations             : 49,425

Small validation configurations: 3


,dataset_id,feature,target,feature_type,strategy_id,missingness_mechanism,missingness_rate,seed,n_observed,n_evaluated,...,rmse,r2,accuracy,macro_f1,weighted_f1,wasserstein,js_divergence,runtime_seconds,status,error
0,adult_income,age,income,numerical,mean,MCAR,0.1,890935,29305,3256,...,13.657815,-0.001227,NaN,NaN,NaN,11.168104,NaN,0.008813,SUCCESS,None
1,adult_income,age,income,numerical,median,MCAR,0.1,890935,29305,3256,...,13.796932,-0.021728,NaN,NaN,NaN,11.142199,NaN,0.008603,SUCCESS,None
2,adult_income,age,income,numerical,constant,MCAR,0.1,890935,29305,3256,...,41.330881,-8.168947,NaN,NaN,NaN,39.011978,NaN,0.007461,SUCCESS,None



Small validation: PASSED

Starting full candidate evaluation...
Completed 250/49,425 | SUCCESS=208 | FAILED=42 | Elapsed=2.22 min
Completed 500/49,425 | SUCCESS=417 | FAILED=83 | Elapsed=4.60 min
Completed 750/49,425 | SUCCESS=626 | FAILED=124 | Elapsed=6.80 min
Completed 1,000/49,425 | SUCCESS=822 | FAILED=178 | Elapsed=8.42 min
Completed 1,250/49,425 | SUCCESS=1,000 | FAILED=250 | Elapsed=9.83 min
Completed 1,500/49,425 | SUCCESS=1,188 | FAILED=312 | Elapsed=11.42 min
Completed 1,750/49,425 | SUCCESS=1,396 | FAILED=354 | Elapsed=13.36 min
Completed 2,000/49,425 | SUCCESS=1,604 | FAILED=396 | Elapsed=15.49 min
Completed 2,250/49,425 | SUCCESS=1,812 | FAILED=438 | Elapsed=17.48 min
Completed 2,500/49,425 | SUCCESS=2,000 | FAILED=500 | Elapsed=18.98 min
Completed 2,750/49,425 | SUCCESS=2,179 | FAILED=571 | Elapsed=20.33 min
Completed 3,000/49,425 | SUCCESS=2,376 | FAILED=624 | Elapsed=22.18 min
Completed 3,250/49,425 | SUCCESS=2,584 | FAILED=666 | Elapsed=24.17 min


In [ ]:
# ============================================================
# NOTEBOOK 06.22 — EVALUATION STATUS SUMMARY
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.22")
print("EVALUATION STATUS SUMMARY")
print("=" * 100)


if "EVALUATION_RESULTS_DF" not in globals():
    raise RuntimeError(
        "EVALUATION_RESULTS_DF not found."
    )


status_summary = (
    EVALUATION_RESULTS_DF[
        "status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(
        name="count"
    )
)


display(
    status_summary
)


successful = int(
    (
        EVALUATION_RESULTS_DF[
            "status"
        ]
        == "SUCCESS"
    ).sum()
)

failed = int(
    (
        EVALUATION_RESULTS_DF[
            "status"
        ]
        == "FAILED"
    ).sum()
)


print(
    f"\nSuccessful runs : {successful:,}"
)

print(
    f"Failed runs     : {failed:,}"
)

print(
    f"Total runs      : "
    f"{len(EVALUATION_RESULTS_DF):,}"
)


if failed > 0:

    print("\nFailure summary:")

    failure_summary = (
        EVALUATION_RESULTS_DF[
            EVALUATION_RESULTS_DF[
                "status"
            ] == "FAILED"
        ]
        .groupby(
            "strategy_id"
        )
        .size()
        .reset_index(
            name="failures"
        )
        .sort_values(
            "failures",
            ascending=False
        )
    )

    display(
        failure_summary
    )


print("=" * 100)
print("EVALUATION STATUS SUMMARY : COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.23 — CANDIDATE PERFORMANCE ANALYSIS
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.23")
print("CANDIDATE PERFORMANCE ANALYSIS")
print("=" * 100)


SUCCESS_RESULTS_DF = (
    EVALUATION_RESULTS_DF[
        EVALUATION_RESULTS_DF[
            "status"
        ] == "SUCCESS"
    ]
    .copy()
)


if SUCCESS_RESULTS_DF.empty:
    raise RuntimeError(
        "No successful evaluation results available."
    )


numeric_metrics = [
    "mae",
    "rmse",
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "wasserstein",
    "js_divergence",
    "runtime_seconds"
]


available_metrics = [
    metric
    for metric in numeric_metrics
    if metric in SUCCESS_RESULTS_DF.columns
]


CANDIDATE_PERFORMANCE_DF = (
    SUCCESS_RESULTS_DF
    .groupby(
        [
            "strategy_id",
            "feature_type"
        ],
        dropna=False
    )[available_metrics]
    .agg(
        ["mean", "std", "count"]
    )
    .reset_index()
)


display(
    CANDIDATE_PERFORMANCE_DF
)


print("\nPerformance by strategy:")


strategy_summary = (
    SUCCESS_RESULTS_DF
    .groupby(
        "strategy_id"
    )[available_metrics]
    .mean()
    .reset_index()
)


display(
    strategy_summary
)


ANALYSIS_PATH = (
    RESULTS_DIR /
    "candidate_performance_summary.csv"
)


strategy_summary.to_csv(
    ANALYSIS_PATH,
    index=False
)


print(
    f"\nPerformance summary saved to:\n"
    f"{ANALYSIS_PATH}"
)

print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.24 — LLM RECOMMENDATION EVALUATION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.24")
print("LLM RECOMMENDATION EVALUATION")
print("=" * 100)


if "llm_recommended" not in EVALUATION_PLAN_DF.columns:
    raise RuntimeError(
        "LLM recommendation column not found."
    )


LLM_EVALUATION_DF = (
    EVALUATION_RESULTS_DF
    .merge(
        EVALUATION_PLAN_DF[
            [
                "dataset_id",
                "feature",
                "missingness_mechanism",
                "missingness_rate",
                "seed",
                "llm_recommended"
            ]
        ],
        on=[
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate",
            "seed"
        ],
        how="left"
    )
)


LLM_EVALUATION_DF[
    "llm_match"
] = (
    LLM_EVALUATION_DF[
        "strategy_id"
    ]
    ==
    LLM_EVALUATION_DF[
        "llm_recommended"
    ]
)


LLM_MATCH_RATE = float(
    LLM_EVALUATION_DF[
        "llm_match"
    ].mean()
)


print(
    f"\nLLM recommendation agreement rate: "
    f"{LLM_MATCH_RATE:.4f}"
)


LLM_PERFORMANCE_DF = (
    LLM_EVALUATION_DF
    .groupby(
        "llm_match"
    )[
        [
            "mae",
            "rmse",
            "r2",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "wasserstein",
            "js_divergence",
            "runtime_seconds"
        ]
    ]
    .mean()
)


display(
    LLM_PERFORMANCE_DF
)


LLM_RESULTS_PATH = (
    RESULTS_DIR /
    "llm_recommendation_evaluation.csv"
)


LLM_EVALUATION_DF.to_csv(
    LLM_RESULTS_PATH,
    index=False
)


print(
    f"\nLLM evaluation saved to:\n"
    f"{LLM_RESULTS_PATH}"
)

print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.25 — STATISTICAL SIGNIFICANCE
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.25")
print("STATISTICAL SIGNIFICANCE ANALYSIS")
print("=" * 100)


from scipy.stats import (
    wilcoxon,
    friedmanchisquare
)


SIGNIFICANCE_ALPHA = 0.05


# ------------------------------------------------------------
# Strategy-level metric table
# ------------------------------------------------------------

sig_rows = []


metric_direction = {
    "mae": "lower",
    "rmse": "lower",
    "r2": "higher",
    "accuracy": "higher",
    "macro_f1": "higher",
    "weighted_f1": "higher",
    "wasserstein": "lower",
    "js_divergence": "lower"
}


for metric, direction in metric_direction.items():

    if metric not in SUCCESS_RESULTS_DF.columns:
        continue

    metric_df = SUCCESS_RESULTS_DF[
        [
            "strategy_id",
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate",
            "seed",
            metric
        ]
    ].dropna(
        subset=[metric]
    )

    if metric_df.empty:
        continue

    strategies = sorted(
        metric_df[
            "strategy_id"
        ].unique()
    )

    if len(strategies) < 2:
        continue

    pivot = metric_df.pivot_table(
        index=[
            "dataset_id",
            "feature",
            "missingness_mechanism",
            "missingness_rate",
            "seed"
        ],
        columns="strategy_id",
        values=metric,
        aggfunc="mean"
    ).dropna(
        axis=0,
        how="any"
    )

    if pivot.shape[0] < 2:
        continue

    arrays = [
        pivot[strategy].to_numpy()
        for strategy in strategies
    ]

    try:

        statistic, p_value = (
            friedmanchisquare(
                *arrays
            )
        )

    except Exception:

        statistic = np.nan
        p_value = np.nan


    sig_rows.append({

        "metric": metric,
        "direction": direction,
        "n_strategies": len(strategies),
        "n_matched_cases": len(pivot),
        "friedman_statistic": statistic,
        "p_value": p_value,
        "significant_at_0.05": (
            bool(
                p_value < SIGNIFICANCE_ALPHA
            )
            if np.isfinite(p_value)
            else False
        )

    })


STATISTICAL_SIGNIFICANCE_DF = pd.DataFrame(
    sig_rows
)


display(
    STATISTICAL_SIGNIFICANCE_DF
)


SIG_PATH = (
    RESULTS_DIR /
    "statistical_significance.csv"
)


STATISTICAL_SIGNIFICANCE_DF.to_csv(
    SIG_PATH,
    index=False
)


print(
    f"\nStatistical significance results saved to:\n"
    f"{SIG_PATH}"
)

print("=" * 100)

In [ ]:
# ============================================================
# NOTEBOOK 06.26 — ADAPTIVE UTILITY-BASED SELECTION
# ============================================================

print("=" * 100)
print("AIR-LLM — NOTEBOOK 06.26")
print("ADAPTIVE UTILITY-BASED SELECTION")
print("=" * 100)


# ------------------------------------------------------------
# Utility dimensions
# ------------------------------------------------------------

UTILITY_WEIGHTS = {
    "reconstruction": 0.30,
    "distribution": 0.20,
    "dependency": 0.15,
    "downstream": 0.20,
    "efficiency": 0.15
}


def minmax_normalize(
    series,
    higher_is_better=True
):

    series = pd.to_numeric(
        series,
        errors="coerce"
    )

    minimum = series.min()
    maximum = series.max()

    if not np.isfinite(minimum) or not np.isfinite(maximum):
        return pd.Series(
            0.5,
            index=series.index
        )

    if maximum == minimum:
        return pd.Series(
            1.0,
            index=series.index
        )

    if higher_is_better:

        return (
            series - minimum
        ) / (
            maximum - minimum
        )

    return (
        maximum - series
    ) / (
        maximum - minimum
    )


UTILITY_SOURCE_DF = (
    SUCCESS_RESULTS_DF.copy()
)


# ------------------------------------------------------------
# Reconstruction utility
# ------------------------------------------------------------

utility_components = []


if "mae" in UTILITY_SOURCE_DF.columns:

    utility_components.append(
        minmax_normalize(
            UTILITY_SOURCE_DF["mae"],
            higher_is_better=False
        )
    )

if "rmse" in UTILITY_SOURCE_DF.columns:

    utility_components.append(
        minmax_normalize(
            UTILITY_SOURCE_DF["rmse"],
            higher_is_better=False
        )
    )


if utility_components:

    UTILITY_SOURCE_DF[
        "reconstruction_utility"
    ] = pd.concat(
        utility_components,
        axis=1
    ).mean(
        axis=1
    )

else:

    UTILITY_SOURCE_DF[
        "reconstruction_utility"
    ] = 0.0


# ------------------------------------------------------------
# Distribution utility
# ------------------------------------------------------------

distribution_components = []


if "wasserstein" in UTILITY_SOURCE_DF.columns:

    distribution_components.append(
        minmax_normalize(
            UTILITY_SOURCE_DF[
                "wasserstein"
            ],
            higher_is_better=False
        )
    )


if "js_divergence" in UTILITY_SOURCE_DF.columns:

    distribution_components.append(
        minmax_normalize(
            UTILITY_SOURCE_DF[
                "js_divergence"
            ],
            higher_is_better=False
        )
    )


if distribution_components:

    UTILITY_SOURCE_DF[
        "distribution_utility"
    ] = pd.concat(
        distribution_components,
        axis=1
    ).mean(
        axis=1
    )

else:

    UTILITY_SOURCE_DF[
        "distribution_utility"
    ] = 0.0


# ------------------------------------------------------------
# Downstream/reconstruction categorical utility
# ------------------------------------------------------------

downstream_components = []


for metric in [
    "r2",
    "accuracy",
    "macro_f1",
    "weighted_f1"
]:

    if metric in UTILITY_SOURCE_DF.columns:

        values = UTILITY_SOURCE_DF[
            metric
        ]

        if values.notna().any():

            downstream_components.append(
                minmax_normalize(
                    values,
                    higher_is_better=True
                )
            )


if downstream_components:

    UTILITY_SOURCE_DF[
        "downstream_utility"
    ] = pd.concat(
        downstream_components,
        axis=1
    ).mean(
        axis=1
    )

else:

    UTILITY_SOURCE_DF[
        "downstream_utility"
    ] = 0.0


# ------------------------------------------------------------
# Dependency utility
# ------------------------------------------------------------

if "dependency_error" in UTILITY_SOURCE_DF.columns:

    UTILITY_SOURCE_DF[
        "dependency_utility"
    ] = minmax_normalize(
        UTILITY_SOURCE_DF[
            "dependency_error"
        ],
        higher_is_better=False
    )

else:

    UTILITY_SOURCE_DF[
        "dependency_utility"
    ] = 0.5


# ------------------------------------------------------------
# Computational efficiency
# ------------------------------------------------------------

UTILITY_SOURCE_DF[
    "efficiency_utility"
] = minmax_normalize(
    UTILITY_SOURCE_DF[
        "runtime_seconds"
    ],
    higher_is_better=False
)


# ------------------------------------------------------------
# Composite adaptive utility
# ------------------------------------------------------------

UTILITY_SOURCE_DF[
    "adaptive_utility"
] = (

    UTILITY_WEIGHTS[
        "reconstruction"
    ]
    *
    UTILITY_SOURCE_DF[
        "reconstruction_utility"
    ]

    +

    UTILITY_WEIGHTS[
        "distribution"
    ]
    *
    UTILITY_SOURCE_DF[
        "distribution_utility"
    ]

    +

    UTILITY_WEIGHTS[
        "dependency"
    ]
    *
    UTILITY_SOURCE_DF[
        "dependency_utility"
    ]

    +

    UTILITY_WEIGHTS[
        "downstream"
    ]
    *
    UTILITY_SOURCE_DF[
        "downstream_utility"
    ]

    +

    UTILITY_WEIGHTS[
        "efficiency"
    ]
    *
    UTILITY_SOURCE_DF[
        "efficiency_utility"
    ]
)


# ------------------------------------------------------------
# Select best strategy per evaluation context
# ------------------------------------------------------------

SELECTION_KEYS = [
    "dataset_id",
    "feature",
    "missingness_mechanism",
    "missingness_rate",
    "seed"
]


ADAPTIVE_SELECTION_DF = (
    UTILITY_SOURCE_DF
    .sort_values(
        "adaptive_utility",
        ascending=False
    )
    .groupby(
        SELECTION_KEYS,
        as_index=False
    )
    .first()
)


ADAPTIVE_SELECTION_DF = (
    ADAPTIVE_SELECTION_DF[
        SELECTION_KEYS
        +
        [
            "strategy_id",
            "adaptive_utility",
            "reconstruction_utility",
            "distribution_utility",
            "dependency_utility",
            "downstream_utility",
            "efficiency_utility"
        ]
    ]
)


display(
    ADAPTIVE_SELECTION_DF.head(25)
)


ADAPTIVE_RESULTS_PATH = (
    RESULTS_DIR /
    "adaptive_utility_selection.csv"
)


ADAPTIVE_SELECTION_DF.to_csv(
    ADAPTIVE_RESULTS_PATH,
    index=False
)


print(
    f"\nAdaptive selections: "
    f"{len(ADAPTIVE_SELECTION_DF):,}"
)

print(
    f"Results saved to:\n"
    f"{ADAPTIVE_RESULTS_PATH}"
)

print("=" * 100)
print("ADAPTIVE UTILITY SELECTION : COMPLETE")
print("=" * 100)